# Notebook 20 - Empirical Event-Weight API

## Exercise 5: reusable estimator for canonical USDJPY target-event histories

# 1. Purpose and contract

Notebook 20 packages the frozen N16-N19 workflow as a reusable API. It inherits the frozen Notebook 18 same-model-day event mapping and uses target_model_day in all long-form target output. The N20 equivalence suite validates faithful reproduction of the frozen upstream methodology; it does not independently prove that the economic model or estimand is structurally correct.

## 2. Imports, paths, frozen configuration, and reference inputs

In [1]:
from pathlib import Path
import math, random
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mutual_info_score
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import display
# N20 is launched from the project directory containing Data/processed.
# Set this only when deliberately launching the notebook from elsewhere.
PROJECT_DIR_OVERRIDE = None
PROJECT_DIR = (
    Path(PROJECT_DIR_OVERRIDE).expanduser().resolve()
    if PROJECT_DIR_OVERRIDE is not None
    else Path.cwd().resolve()
)
DATA_DIR = PROJECT_DIR / 'Data'
PROCESSED = DATA_DIR / 'processed'
if not PROCESSED.is_dir():
    raise RuntimeError(
        'N20 resolved no canonical Data/processed directory. '
        f'current working directory={Path.cwd().resolve()}; '
        f'PROJECT_DIR={PROJECT_DIR}; expected={PROCESSED}. '
        'Launch N20 from the project root or set PROJECT_DIR_OVERRIDE explicitly.'
    )
torch.set_num_threads(1); DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available(): torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
RF_GRID={"RF_01":{"max_depth":8,"min_samples_leaf":5},"RF_02":{"max_depth":8,"min_samples_leaf":10},"RF_03":{"max_depth":None,"min_samples_leaf":5},"RF_04":{"max_depth":None,"min_samples_leaf":10}}; GB_GRID={"GB_01":{"n_estimators":200,"learning_rate":.03},"GB_02":{"n_estimators":200,"learning_rate":.05},"GB_03":{"n_estimators":400,"learning_rate":.03},"GB_04":{"n_estimators":400,"learning_rate":.05}}; MLP_GRID={"MLP_01":([32],1e-3),"MLP_02":([32],3e-4),"MLP_03":([32,16],1e-3),"MLP_04":([32,16],3e-4)}; TR_GRID={"TR_01":((16,2,32),1e-3),"TR_02":((16,2,32),3e-4),"TR_03":((24,4,48),1e-3),"TR_04":((24,4,48),3e-4)}
FROZEN_TARGET_FLOORS = {'R': 0.0003202993332501, 'I': 0.0015767239436135}

def admissible_forecast(raw_forecast, target):
    raw = np.asarray(raw_forecast, dtype=float)
    return np.where(np.isfinite(raw), np.maximum(raw, FROZEN_TARGET_FLOORS[target]), np.nan)

CONFIG={"max_lag":12,"states":["Q","R","I"],"features":[f"{s}_lag{j}" for j in range(12,0,-1) for s in "QRI"],"gc_alpha":.05,"tdmi_bins":10,"tdmi_surrogates":1000,"neural_seeds":[19,119,219],"max_epochs":400,"patience":40,"min_delta":1e-5,"batch_size":64,"rf_grid":RF_GRID,"gb_grid":GB_GRID,"mlp_grid":MLP_GRID,"transformer_grid":TR_GRID,"bootstrap_reps":10000,"bootstrap_seed":18,"transformer_max_norm":1.0,"mlp_gradient_clipping":"none","equivalence_tolerances":{"parametric":1e-10,"tree":1e-10,"neural":1e-6}}
def authoritative_specification_row(path):
    table = pd.read_csv(path)
    valid = table.notna().any(axis=1) & ~table.astype(str).apply(lambda row: row.str.contains("<<<<<<<|=======|>>>>>>>", regex=True).any(), axis=1)
    if not valid.any():
        raise ValueError(f"No authoritative row in {path}")
    candidates = table.loc[valid].copy().reset_index(drop=True)
    # Repeated exports may differ only in run provenance; compare the substantive frozen contract.
    contract_columns = [column for column in candidates.columns if column.lower() not in {'run_id', 'runid', 'timestamp', 'created_at'}]
    contracts = candidates[contract_columns].drop_duplicates().reset_index(drop=True)
    if len(contracts) != 1:
        raise ValueError(f'Conflicting populated specification contracts in {path}: {contracts.to_dict("records")}')
    return candidates.iloc[0]

UPSTREAM_FILES = {
    'N16': ('16_empirical_analysis_panel.csv', '16_primary_specification.csv', '16_model_specification.csv'),
    'N17': ('17_forecasts_validation.csv', '17_forecasts_test.csv'),
    'N18': ('18_primary_specification.csv', '18_target_specific_event_mapping.csv', '18_event_response_weights.csv'),
    'N19': ('19_selected_model_specifications.csv', '19_ordinary_response_ratios.csv', '19_eventless_forecasts_validation.csv', '19_eventless_forecasts_test.csv', '19_event_response_weights.csv'),
}
FROZEN_SPLITS = {
    'train': (pd.Timestamp('2003-05-05'), pd.Timestamp('2017-03-27')),
    'validation': (pd.Timestamp('2017-03-28'), pd.Timestamp('2021-11-12')),
    'test': (pd.Timestamp('2021-11-15'), pd.Timestamp('2026-07-02')),
}

def _preflight_table(path):
    frame = pd.read_csv(path)
    # N18's frozen export also names the canonical admissible alias background_adm.
    if path.name == '18_event_response_weights.csv' and 'background_forecast' not in frame and 'admissible_background_forecast' in frame:
        frame['background_forecast'] = frame['admissible_background_forecast']
    return frame, {
        'filename': path.name, 'resolved_path': str(path.resolve()),
        'exists': True, 'file_size_bytes': int(path.stat().st_size),
        'n_rows': int(len(frame)), 'n_columns': int(len(frame.columns)),
    }

def _preflight_require_columns(frame, required, label, failures):
    missing = sorted(set(required).difference(frame.columns))
    if missing:
        failures.append(f'{label} missing required columns: {missing}')
    if frame.columns.duplicated().any():
        failures.append(f'{label} contains duplicate column names')

def _preflight_unique(frame, keys, label, failures):
    if set(keys).issubset(frame.columns) and frame.duplicated(keys).any():
        failures.append(f'{label} contains duplicate stable keys {keys}')

def preflight_n20_inputs(processed=PROCESSED):
    """Validate frozen inputs before any empirical fit or equivalence reconstruction."""
    failures, checks, tables, manifest_rows = [], [], {}, []
    checks.append({'check': 'project directory', 'status': 'PASS', 'detail': str(PROJECT_DIR)})
    for source_notebook, filenames in UPSTREAM_FILES.items():
        for filename in filenames:
            path = processed / filename
            if not path.is_file():
                failures.append(f'Missing required {source_notebook} input: {path}')
                manifest_rows.append({'source_notebook': source_notebook, 'filename': filename, 'resolved_path': str(path.resolve()), 'n_rows': np.nan, 'n_columns': np.nan, 'status': 'MISSING'})
                continue
            try:
                tables[filename], row = _preflight_table(path)
                row.update({'source_notebook': source_notebook, 'status': 'PASS'})
                manifest_rows.append(row)
            except Exception as exc:
                failures.append(f'Cannot read {path}: {exc}')
                manifest_rows.append({'source_notebook': source_notebook, 'filename': filename, 'resolved_path': str(path.resolve()), 'n_rows': np.nan, 'n_columns': np.nan, 'status': 'UNREADABLE'})
    checks.append({'check': 'required files', 'status': 'PASS' if not failures else 'FAIL', 'detail': '; '.join(failures) or 'all required CSVs are readable'})
    panel = tables.get('16_empirical_analysis_panel.csv')
    spec16_table = tables.get('16_primary_specification.csv')
    spec18_table = tables.get('18_primary_specification.csv')
    if panel is not None and spec16_table is not None:
        spec16 = authoritative_specification_row(processed / '16_primary_specification.csv')
        iv_column = str(spec16['iv_column'])
        required_panel = {'model_day', 'sample_split', 'squared_return', 'realised_variance_ann_252', iv_column}
        _preflight_require_columns(panel, required_panel, 'N16 panel', failures)
        model_day = pd.to_datetime(panel.get('model_day'), errors='coerce')
        duplicate_days = model_day[model_day.notna() & model_day.duplicated(keep=False)]
        panel_detail = (f'rows={len(panel)}, non_null_model_day={int(model_day.notna().sum())}, '
            f'missing_or_NaT={int(model_day.isna().sum())}, unique_days={int(model_day.nunique())}, '
            f'duplicated_day_rows={int(duplicate_days.size)}, first={model_day.min()}, last={model_day.max()}, '
            f'ordered={bool(model_day.is_monotonic_increasing)}')
        if model_day.isna().any():
            markers = panel.loc[model_day.isna()].head(12).to_dict('records')
            failures.append(f'N16 panel has {int(model_day.isna().sum())} missing/NaT model_day rows; examples={markers}')
        if duplicate_days.size:
            duplicate_examples = panel.loc[model_day.isin(duplicate_days.unique())].head(24).to_dict('records')
            failures.append(f'N16 panel has {int(duplicate_days.size)} duplicated model_day rows; examples={duplicate_examples}')
        if not model_day.is_monotonic_increasing:
            failures.append('N16 panel is not chronologically ordered in its source CSV')
        checks.extend([
            {'check': 'N16 panel schema', 'status': 'PASS' if required_panel.issubset(panel.columns) and not panel.columns.duplicated().any() else 'FAIL', 'detail': iv_column},
            {'check': 'N16 model_day integrity', 'status': 'PASS' if model_day.notna().all() and model_day.is_unique else 'FAIL', 'detail': panel_detail},
            {'check': 'N16 chronology', 'status': 'PASS' if model_day.is_monotonic_increasing else 'FAIL', 'detail': panel_detail},
        ])
        if 'sample_split' in panel:
            split_counts = panel['sample_split'].value_counts(dropna=False).to_dict()
            unexpected = set(panel['sample_split'].dropna()).difference(FROZEN_SPLITS)
            contradictory = []
            for split, (start, end) in FROZEN_SPLITS.items():
                part = model_day[panel['sample_split'].eq(split)]
                if ((part < start) | (part > end)).any(): contradictory.append(split)
            if panel['sample_split'].isna().any() or unexpected or contradictory:
                failures.append(f'N16 panel invalid sample_split values/counts={split_counts}; unexpected={sorted(unexpected)}; contradictory={contradictory}')
            checks.append({'check': 'split chronology', 'status': 'PASS' if not (panel['sample_split'].isna().any() or unexpected or contradictory) else 'FAIL', 'detail': str(split_counts)})
        if (int(spec16['selected_realised_order']), int(spec16['selected_implied_order'])) != (1, 12):
            failures.append('N16 frozen structure does not retain selected orders R=1 and I=12')
        checks.append({'check': 'N16 frozen structure', 'status': 'PASS' if (int(spec16['selected_realised_order']), int(spec16['selected_implied_order'])) == (1, 12) else 'FAIL', 'detail': 'R=1, I=12 required'})
    if spec18_table is not None:
        spec18 = authoritative_specification_row(processed / '18_primary_specification.csv')
        aligned = str(spec18.get('implied_event_alignment', '')) == 'SAME_EMPIRICAL_MODEL_DAY_AS_EVENT'
        timing = str(spec18.get('bloomberg_snapshot_time_identified', '')).lower() in {'false', '0'}
        if not aligned or not timing: failures.append('N18 timing metadata must retain same-model-day mapping and an unresolved Bloomberg snapshot time')
        checks.append({'check': 'N18 timing contract', 'status': 'PASS' if aligned and timing else 'FAIL', 'detail': str(spec18.get('alignment_interpretation', 'missing'))})
    mapping = tables.get('18_target_specific_event_mapping.csv')
    if mapping is not None:
        _preflight_require_columns(mapping, {'event_id', 'event_label', 'target', 'realised_model_day', 'implied_model_day', 'target_model_day', 'sample_split'}, 'N18 event mapping', failures)
        dates = {name: pd.to_datetime(mapping[name], errors='coerce') for name in ('realised_model_day', 'implied_model_day', 'target_model_day') if name in mapping}
        mapping_ok = set(mapping.get('target', pd.Series(dtype=object)).dropna()).issubset({'R', 'I'}) and set(mapping.get('sample_split', pd.Series(dtype=object)).dropna()).issubset(FROZEN_SPLITS) and dates.get('realised_model_day', pd.Series(dtype='datetime64[ns]')).eq(dates.get('implied_model_day', pd.Series(dtype='datetime64[ns]'))).all()
        _preflight_unique(mapping, ['event_id', 'target', 'target_model_day'], 'N18 event mapping', failures)
        if not mapping_ok: failures.append('N18 mapping violates valid target labels or realised/implied model-day equality')
        checks.append({'check': 'N18 event mapping', 'status': 'PASS' if mapping_ok else 'FAIL', 'detail': f'rows={len(mapping)}'})
    n18_weights = tables.get('18_event_response_weights.csv')
    if n18_weights is not None:
        _preflight_require_columns(n18_weights, {'event_id', 'event_label', 'target', 'target_model_day', 'sample_split', 'raw_background_forecast', 'admissible_background_forecast', 'background_forecast', 'forecast_floor', 'floor_activated', 'event_weight_raw', 'event_weight_adm', 'event_weight', 'raw_weight_defined', 'weight_defined'}, 'N18 event-response weights', failures)
        _preflight_unique(n18_weights, ['event_id', 'target', 'target_model_day', 'event_label'], 'N18 event-response weights', failures)
        checks.append({'check': 'N18 event-weight reference', 'status': 'PASS' if not any('N18 event-response weights' in item for item in failures) else 'FAIL', 'detail': f'splits={sorted(n18_weights.get("sample_split", pd.Series(dtype=object)).dropna().unique().tolist())}'})
    for filename in ('17_forecasts_validation.csv', '17_forecasts_test.csv'):
        frame = tables.get(filename)
        if frame is None: continue
        _preflight_require_columns(frame, {'model_day', 'sample_split', 'target', 'model', 'forecast_scheme', 'raw_forecast', 'admissible_forecast', 'train_positive_floor', 'floor_activated'}, filename, failures)
        split = 'validation' if 'validation' in filename else 'test'
        scheme = 'fixed_train' if split == 'validation' else 'fixed_train_validation'
        for target, model in {'R': 'R_PRIMARY_AR1_IV', 'I': 'I_PRIMARY_FULL_12'}.items():
            part = frame.loc[frame.get('target', pd.Series(dtype=object)).eq(target) & frame.get('model', pd.Series(dtype=object)).eq(model) & frame.get('forecast_scheme', pd.Series(dtype=object)).eq(scheme)]
            _preflight_unique(part, ['model_day'], f'N17 {split} {target} primary reference', failures)
            if part.empty: failures.append(f'N17 {split} primary reference missing for {target}/{model}/{scheme}')
        checks.append({'check': f'N17 {split} primary references', 'status': 'PASS' if not any(f'N17 {split}' in item for item in failures) else 'FAIL', 'detail': scheme})
    n19_spec = tables.get('19_selected_model_specifications.csv')
    if n19_spec is not None:
        _preflight_require_columns(n19_spec, {'model', 'target', 'selected_parent_blocks', 'selected_lag_order', 'hyperparameter_candidate_id', 'final_fixed_epochs'}, 'N19 specification', failures)
        expected = {('R', 'RF'): (('R', 'I'), 1), ('R', 'GB'): (('R', 'I'), 2), ('R', 'MLP'): (('R', 'I'), 1), ('R', 'TRANSFORMER'): (('R', 'I'), 1), ('I', 'RF'): (('I',), 2), ('I', 'GB'): (('I',), 2), ('I', 'MLP'): (('I', 'Q'), 4), ('I', 'TRANSFORMER'): (('I', 'Q', 'R'), 5)}
        observed = {(row.target, row.model): (tuple(str(row.selected_parent_blocks).split('|')), int(row.selected_lag_order)) for row in n19_spec.itertuples(index=False)}
        valid_n19 = len(n19_spec) == 8 and not n19_spec.duplicated(['target', 'model']).any() and observed == expected
        if not valid_n19: failures.append(f'N19 frozen specification coverage/structure differs from canonical export: {observed}')
        checks.append({'check': 'N19 specification coverage', 'status': 'PASS' if valid_n19 else 'FAIL', 'detail': '8 model-target specifications'})
    n19_required = {
        '19_ordinary_response_ratios.csv': ({'model_day', 'target_model_day', 'sample_split', 'forecast_stage', 'target', 'model', 'raw_forecast', 'admissible_forecast', 'background_forecast', 'forecast_floor', 'floor_activated', 'ordinary_response_ratio_raw', 'ordinary_response_ratio_adm', 'response_ratio', 'raw_positive_background', 'background_positive', 'equation_eligible'}, ['model_day', 'target', 'model', 'sample_split', 'forecast_stage']),
        '19_eventless_forecasts_validation.csv': ({'model_day', 'target_model_day', 'sample_split', 'forecast_stage', 'target', 'model', 'raw_background_forecast', 'admissible_background_forecast', 'background_forecast', 'forecast_floor', 'floor_activated'}, ['model_day', 'target', 'model', 'sample_split', 'forecast_stage']),
        '19_eventless_forecasts_test.csv': ({'model_day', 'target_model_day', 'sample_split', 'forecast_stage', 'target', 'model', 'raw_background_forecast', 'admissible_background_forecast', 'background_forecast', 'forecast_floor', 'floor_activated'}, ['model_day', 'target', 'model', 'sample_split', 'forecast_stage']),
        '19_event_response_weights.csv': ({'event_id', 'event_label', 'target_model_day', 'sample_split', 'target', 'model', 'raw_background_forecast', 'admissible_background_forecast', 'background_forecast', 'forecast_floor', 'floor_activated', 'event_weight_raw', 'event_weight_adm', 'event_weight', 'raw_weight_defined', 'weight_defined'}, ['event_id', 'target_model_day', 'target', 'model', 'event_label']),
    }
    for filename, (required, keys) in n19_required.items():
        frame = tables.get(filename)
        if frame is not None:
            _preflight_require_columns(frame, required, filename, failures)
            _preflight_unique(frame, keys, filename, failures)
    checks.append({'check': 'N19 reference schemas and uniqueness', 'status': 'PASS' if not any('19_' in item for item in failures) else 'FAIL', 'detail': 'ordinary, eventless, and event-weight reference keys'})
    split_support = {filename: sorted(frame['sample_split'].dropna().unique().tolist()) for filename, frame in tables.items() if 'sample_split' in frame.columns}
    checks.append({'check': 'reference split support', 'status': 'PASS', 'detail': str(split_support)})
    # Canonical aliases must carry the admissible production quantities.
    for name, raw, admissible, canonical in [('18_event_response_weights.csv', 'raw_background_forecast', 'admissible_background_forecast', 'background_forecast'), ('19_ordinary_response_ratios.csv', 'raw_forecast', 'admissible_forecast', 'background_forecast'), ('19_eventless_forecasts_validation.csv', 'raw_background_forecast', 'admissible_background_forecast', 'background_forecast'), ('19_eventless_forecasts_test.csv', 'raw_background_forecast', 'admissible_background_forecast', 'background_forecast'), ('19_event_response_weights.csv', 'raw_background_forecast', 'admissible_background_forecast', 'background_forecast')]:
        frame = tables.get(name)
        if frame is not None and {admissible, canonical}.issubset(frame.columns) and not np.allclose(frame[admissible], frame[canonical], equal_nan=True):
            failures.append(f'{name} canonical background is not the admissible background')
    for name in ('18_event_response_weights.csv', '19_event_response_weights.csv'):
        frame = tables.get(name)
        if frame is not None and {'event_weight', 'event_weight_adm'}.issubset(frame.columns) and not np.allclose(frame['event_weight'], frame['event_weight_adm'], equal_nan=True):
            failures.append(f'{name} canonical event weight is not the admissible event weight')
    ordinary = tables.get('19_ordinary_response_ratios.csv')
    if ordinary is not None and {'response_ratio', 'ordinary_response_ratio_adm'}.issubset(ordinary.columns) and not np.allclose(ordinary['response_ratio'], ordinary['ordinary_response_ratio_adm'], equal_nan=True):
        failures.append('N19 ordinary canonical response ratio is not admissible')
    input_manifest = pd.DataFrame(manifest_rows)
    preflight_summary = pd.DataFrame(checks)
    display(input_manifest)
    display(preflight_summary)
    if failures:
        print('N20 PREFLIGHT: FAIL')
        raise RuntimeError('N20 preflight failed before model fitting:\n- ' + '\n- '.join(failures))
    print('N20 PREFLIGHT: PASS')
    return {'tables': tables, 'input_manifest': input_manifest, 'summary': preflight_summary, 'split_support': split_support}

PREFLIGHT = preflight_n20_inputs()
spec16 = authoritative_specification_row(PROCESSED / '16_primary_specification.csv')
spec18 = authoritative_specification_row(PROCESSED / '18_primary_specification.csv')
IV_COLUMN = str(spec16['iv_column'])
raw_model_panel = PREFLIGHT['tables']['16_empirical_analysis_panel.csv'].copy()
reference_mapping = PREFLIGHT['tables']['18_target_specific_event_mapping.csv'].copy()
reference_n18 = PREFLIGHT['tables']['18_event_response_weights.csv'].copy()
reference_n19 = PREFLIGHT['tables']['19_event_response_weights.csv'].copy()
reference_n19_spec = PREFLIGHT['tables']['19_selected_model_specifications.csv'].copy()
reference_n19_ordinary = PREFLIGHT['tables']['19_ordinary_response_ratios.csv'].copy()
REFERENCE_SPLIT_SUPPORT = PREFLIGHT['split_support']


,filename,resolved_path,exists,file_size_bytes,n_rows,n_columns,source_notebook,status
0,16_empirical_analysis_panel.csv,C:\Users\Rajiv Nawal\OneDrive\Documents\GITREP...,True,1154018,6044,12,N16,PASS
1,16_primary_specification.csv,C:\Users\Rajiv Nawal\OneDrive\Documents\GITREP...,True,1451,9,23,N16,PASS
2,16_model_specification.csv,C:\Users\Rajiv Nawal\OneDrive\Documents\GITREP...,True,1451,9,23,N16,PASS
3,17_forecasts_validation.csv,C:\Users\Rajiv Nawal\OneDrive\Documents\GITREP...,True,3737984,11340,22,N17,PASS
4,17_forecasts_test.csv,C:\Users\Rajiv Nawal\OneDrive\Documents\GITREP...,True,4534187,13518,23,N17,PASS
5,18_primary_specification.csv,C:\Users\Rajiv Nawal\OneDrive\Documents\GITREP...,True,2163,1,50,N18,PASS
6,18_target_specific_event_mapping.csv,C:\Users\Rajiv Nawal\OneDrive\Documents\GITREP...,True,1124243,2828,40,N18,PASS
7,18_event_response_weights.csv,C:\Users\Rajiv Nawal\OneDrive\Documents\GITREP...,True,1774852,2828,62,N18,PASS
8,19_selected_model_specifications.csv,C:\Users\Rajiv Nawal\OneDrive\Documents\GITREP...,True,773,8,9,N19,PASS
9,19_ordinary_response_ratios.csv,C:\Users\Rajiv Nawal\OneDrive\Documents\GITREP...,True,6018649,23377,21,N19,PASS


,check,status,detail
0,project directory,PASS,C:\Users\Rajiv Nawal\OneDrive\Documents\GITREP...
1,required files,PASS,all required CSVs are readable
2,N16 panel schema,PASS,iv_model_var
3,N16 model_day integrity,PASS,"rows=6044, non_null_model_day=6044, missing_or..."
4,N16 chronology,PASS,"rows=6044, non_null_model_day=6044, missing_or..."
5,split chronology,PASS,"{'train': 3626, 'validation': 1209, 'test': 1209}"
6,N16 frozen structure,PASS,"R=1, I=12 required"
7,N18 timing contract,PASS,empirical model-day convention; Bloomberg dail...
8,N18 event mapping,PASS,rows=2828
9,N18 event-weight reference,PASS,"splits=['test', 'train', 'validation']"


N20 PREFLIGHT: PASS


## Canonical packaged workflow

Step 1 estimates or loads the N16 parametric structure. Step 2 produces event-unaware ordinary forecasts. Step 3 refits models after target-event masking. Step 4 applies the dynamic N19 flexible specifications. FULL mode can discover structures and tune hyperparameters; REUSE mode accepts the frozen structures and refits parameters and scalers only.

For every model family, the common forecaster contract maps the available history into a target forecast:

$$
\mathcal H_{t-1}
=
\{Q_{t-j}, R_{t-j}, I_{t-j}\}.
$$

$$
\widehat Y_t
=
f_M(\mathcal H_{t-1}; \widehat\theta_M).
$$

Ordinary backgrounds fit on the intact calibration sample. Eventless backgrounds use the same forecaster but exclude only target-event outcome rows from calibration; the observed historical states remain available as later lags:

$$
\widehat\theta_M^0
=
\operatorname{Fit}\left(\{t:t\in\mathcal C,\ t\notin\mathcal E\}\right),
$$

$$
\widehat F_t^{M,Y,0}
=
f_M(\mathcal H_{t-1}; \widehat\theta_M^0),
$$

$$
\widehat F_{M,t}^{Y,0,\mathrm{adm}}
=
\max\left(\widehat F_{M,t}^{Y,0,\mathrm{raw}}, \epsilon_Y\right),
$$

where the frozen target-specific floors are $\epsilon_R = 0.0003202993332501$ and $\epsilon_I = 0.0015767239436135$. Raw forecasts and backgrounds are retained unchanged for diagnostics; raw response ratios and raw event weights are defined only when their raw denominator is finite and strictly positive.

The canonical production event weight uses the admissible background:

$$
\widehat\omega_{M,t}^{Y,\mathrm{adm}}
=
\frac{Y_t}{\widehat F_{M,t}^{Y,0,\mathrm{adm}}}.
$$

The floor does not alter model fitting. Observed targets and event weights are not clipped.


## Timing and downstream contract

17:00 New York is the empirical model-day boundary. Events use the same-model-day mapping inherited from frozen Notebook 18, so realised and implied targets share one model day. The exact Bloomberg daily ATM implied-volatility snapshot or fixing time is unresolved from the available metadata. No Employment D-1 rule, BoJ D-1 rule, or family-specific timing offset is used. Ordinary-ratio exports are the event-unaware Step-2 forecasts required by Notebook 21; eventless forecasts are used only for event weights.


In [2]:
# Canonical dynamic N19-compatible flexible estimator.
# This cell intentionally replaces the retired fixed-width prototype above.
import ast

MODEL_FAMILIES = ('RF', 'GB', 'MLP', 'TRANSFORMER')
TARGETS = ('R', 'I')
PARENT_SETS = {'R': (('R',), ('R', 'Q'), ('R', 'I'), ('R', 'Q', 'I')), 'I': (('I',), ('I', 'Q'), ('I', 'R'), ('I', 'Q', 'R'))}
GRIDS = {'RF': RF_GRID, 'GB': GB_GRID, 'MLP': MLP_GRID, 'TRANSFORMER': TR_GRID}
PROBE_CONFIG = {family: next(iter(GRIDS[family])) for family in MODEL_FAMILIES}
RF_FIXED = {'n_estimators': 500, 'max_features': .5, 'criterion': 'squared_error', 'bootstrap': True, 'random_state': 19, 'n_jobs': -1}
GB_FIXED = {'max_depth': 2, 'min_samples_leaf': 10, 'subsample': 1.0, 'loss': 'squared_error', 'random_state': 19}
ALIGNMENT_VERSION = 'MODEL_DAY_1700_NY_SAME_MODEL_DAY'
CONFIG.update({'max_lag': 12, 'neural_seeds': (19, 119, 219), 'max_epochs': 400, 'patience': 40, 'min_delta': 1e-5, 'batch_size': 64, 'parsimony_band': 1.01})

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def flexible_feature_names(parent_blocks, lag_order):
    parents = tuple(parent_blocks)
    if not parents or any(state not in CONFIG['states'] for state in parents): raise ValueError('invalid parent blocks')
    if len(set(parents)) != len(parents) or not 1 <= int(lag_order) <= CONFIG['max_lag']: raise ValueError('invalid lag order')
    # Final N19 uses oldest-to-newest lags, then parent order, so lag 1 is the final Transformer token.
    return [f'{state}_lag{lag}' for lag in range(int(lag_order), 0, -1) for state in parents]

def prepare_model_panel(data, config=CONFIG):
    panel = data.copy()
    if 'model_day' not in panel.columns:
        raise ValueError('16_empirical_analysis_panel.csv requires model_day')
    panel['model_day'] = pd.to_datetime(panel['model_day'], errors='coerce')
    if panel['model_day'].isna().any():
        raise ValueError(f"16_empirical_analysis_panel.csv contains {int(panel['model_day'].isna().sum())} missing/NaT model_day rows")
    duplicate_days = panel.loc[panel['model_day'].duplicated(keep=False), 'model_day'].dt.strftime('%Y-%m-%d').unique().tolist()
    if duplicate_days:
        raise ValueError(f"16_empirical_analysis_panel.csv contains {len(duplicate_days)} duplicated model_day values: {duplicate_days[:12]}")
    if not panel['model_day'].is_monotonic_increasing:
        raise ValueError('Canonical N16 panel is not chronologically ordered in source CSV; repair N16 upstream rather than sorting in N20')
    panel['Q'] = pd.to_numeric(panel['squared_return'], errors='coerce')
    panel['R'] = pd.to_numeric(panel['realised_variance_ann_252'], errors='coerce')
    panel['I'] = pd.to_numeric(panel[IV_COLUMN], errors='coerce')
    for state in CONFIG['states']:
        for lag in range(1, config['max_lag'] + 1): panel[f'{state}_lag{lag}'] = panel[state].shift(lag)
    return panel

def _normalise_events(events, panel, name):
    needed = {'event_id', 'event_label'}
    if not needed.issubset(events.columns): raise ValueError(f'{name} requires event_id and event_label')
    x = events.copy()
    # sample_split is derived from the authoritative panel and must be idempotent on re-normalisation.
    x = x.drop(columns=[column for column in ('sample_split', 'sample_split_x', 'sample_split_y') if column in x.columns])
    if 'event_timestamp_utc' not in x and 'event_timestamp' in x: x['event_timestamp_utc'] = x['event_timestamp']
    if 'realised_model_day' not in x:
        if 'model_day' in x: x['realised_model_day'] = x['model_day']
        elif 'event_timestamp_utc' in x:
            timestamps = pd.to_datetime(x['event_timestamp_utc'], utc=True, errors='coerce')
            sessions = panel[['model_day']].copy(); sessions['open'] = (sessions['model_day'] - pd.Timedelta(days=1) + pd.Timedelta(hours=17)).dt.tz_localize('America/New_York'); sessions['close'] = (sessions['model_day'] + pd.Timedelta(hours=17)).dt.tz_localize('America/New_York')
            x['realised_model_day'] = [sessions.loc[(sessions['open'] <= timestamp) & (timestamp < sessions['close']), 'model_day'].iloc[0] if pd.notna(timestamp) and ((sessions['open'] <= timestamp) & (timestamp < sessions['close'])).any() else pd.NaT for timestamp in timestamps]
        else: raise ValueError(f'{name} requires a model day or timestamp')
    x['realised_model_day'] = pd.to_datetime(x['realised_model_day'], errors='coerce')
    if x['event_id'].isna().any() or x['event_id'].duplicated().any() or x['realised_model_day'].isna().any(): raise ValueError(f'{name} has invalid or duplicate event identifiers')
    if not x['realised_model_day'].isin(panel['model_day']).all(): raise ValueError(f'{name} has days outside panel support')
    # Corrected N18 production convention: retained I and R occurrences share the Bloomberg model day.
    x['implied_model_day'] = x['realised_model_day']
    if 'event_timestamp_utc' not in x: x['event_timestamp_utc'] = pd.NaT
    x['event_timestamp_utc'] = pd.to_datetime(x['event_timestamp_utc'], utc=True, errors='coerce')
    normalised = x.merge(panel[['model_day', 'sample_split']], left_on='realised_model_day', right_on='model_day', how='left', validate='many_to_one').drop(columns='model_day')
    assert 'sample_split' in normalised.columns
    assert not {'sample_split_x', 'sample_split_y'}.intersection(normalised.columns)
    return normalised

def target_event_mapping(event_history, panel):
    events = _normalise_events(event_history, panel, 'event_history')
    rows = []
    for target in TARGETS:
        part = events.copy(); part['target'] = target; part['target_model_day'] = part['realised_model_day']; rows.append(part)
    mapped = pd.concat(rows, ignore_index=True)
    mapped['n_target_occurrences'] = mapped.groupby(['target', 'target_model_day'])['event_id'].transform('size')
    mapped['is_overlap'] = mapped['n_target_occurrences'].gt(1)
    return mapped.sort_values(['target', 'target_model_day', 'event_id']).reset_index(drop=True)

def _require_event_subset(history, evaluation):
    known = history.set_index('event_id')[['event_label', 'realised_model_day']]
    if not set(evaluation['event_id']).issubset(known.index): raise ValueError('evaluation_events must be a subset of event_history')
    check = evaluation.set_index('event_id')[['event_label', 'realised_model_day']]
    if not check.equals(known.loc[check.index]): raise ValueError('evaluation event identity differs from event_history')

def build_eventless_calibration_sample(model_panel, event_history, evaluation_events=None, config=CONFIG):
    panel = prepare_model_panel(model_panel, config)
    history = _normalise_events(event_history, panel, 'event_history')
    evaluation = history.copy() if evaluation_events is None else _normalise_events(evaluation_events, panel, 'evaluation_events')
    _require_event_subset(history, evaluation)
    panel['is_target_event_day'] = panel['model_day'].isin(set(history['realised_model_day']))
    return {'panel': panel, 'event_history': history, 'evaluation_events': evaluation, 'event_mapping': target_event_mapping(history, panel), 'evaluation_mapping': target_event_mapping(evaluation, panel)}

def _model_frame(panel, target, split, parent_blocks, lag_order, common_support=False):
    features = flexible_feature_names(parent_blocks, lag_order)
    columns = [target, *features] if not common_support else [target, *[f'{state}_lag{lag}' for state in CONFIG['states'] for lag in range(1, CONFIG['max_lag'] + 1)]]
    finite = np.isfinite(panel[columns].to_numpy(dtype=float)).all(axis=1)
    return panel.loc[panel['sample_split'].eq(split) & finite].copy()

def validate_flexible_spec(flexible_spec):
    selected = flexible_spec.get('selected', flexible_spec)
    if not isinstance(selected, dict): raise ValueError('flexible_spec must contain a selected dictionary')
    clean = {}
    for key, specification in selected.items():
        target, family = key
        family = {'RANDOM_FOREST': 'RF', 'GRADIENT_BOOSTING': 'GB'}.get(family, family)
        if target not in TARGETS or family not in MODEL_FAMILIES: raise ValueError('unknown flexible target or family')
        item = dict(specification)
        parents = tuple(item.get('parent_blocks', ()))
        lag_order = int(item.get('lag_order', 0))
        if target not in parents or not set(parents).issubset(CONFIG['states']) or len(set(parents)) != len(parents): raise ValueError('specification must include unique own-target history')
        if not 1 <= lag_order <= CONFIG['max_lag']: raise ValueError('lag_order outside canonical support')
        config_id = item.get('config_id', item.get('hyperparameter_candidate_id'))
        if config_id not in GRIDS[family]: raise ValueError('unknown hyperparameter candidate')
        item.update({'target': target, 'model_family': family, 'parent_blocks': parents, 'lag_order': lag_order, 'config_id': config_id, 'configuration': GRIDS[family][config_id]})
        if family in {'MLP', 'TRANSFORMER'}:
            if not pd.notna(item.get('final_epochs')) or int(item['final_epochs']) < 1: raise ValueError('neural reuse specifications require final_epochs')
            item['final_epochs'] = int(item['final_epochs'])
        clean[(target, family)] = item
    if set(clean) != {(target, family) for target in TARGETS for family in MODEL_FAMILIES}: raise ValueError('specification must cover all target-family pairs')
    return {'selected': clean}

def frozen_n19_flexible_spec():
    source = pd.read_csv(PROCESSED / '19_selected_model_specifications.csv')
    selected = {}
    for row in source.itertuples(index=False):
        selected[(row.target, row.model)] = {'parent_blocks': tuple(row.selected_parent_blocks.split('|')), 'lag_order': int(row.selected_lag_order), 'config_id': row.hyperparameter_candidate_id, 'final_epochs': row.final_fixed_epochs}
    return validate_flexible_spec({'selected': selected})

def fit_state_scaler(frame, target, parent_blocks, lag_order):
    scale = {}
    for state in parent_blocks:
        values = frame[[f'{state}_lag{lag}' for lag in range(1, lag_order + 1)]].to_numpy(dtype=np.float64).ravel()
        scale[state] = (float(values.mean()), float(values.std()))
        if not scale[state][1] > 0: raise ValueError('non-positive predictor scale')
    values = frame[target].to_numpy(dtype=np.float64); scale['TARGET'] = (float(values.mean()), float(values.std()))
    if not scale['TARGET'][1] > 0: raise ValueError('non-positive target scale')
    return scale

def _scaled_features(frame, features, scale):
    values = frame[features].to_numpy(dtype=np.float64).copy()
    for index, feature in enumerate(features):
        mean, std = scale[feature.split('_lag')[0]]; values[:, index] = (values[:, index] - mean) / std
    return values

class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden):
        super().__init__(); layers = []; width = int(input_dim)
        for units in hidden: layers.extend([nn.Linear(width, units), nn.ReLU(), nn.Dropout(.1)]); width = units
        layers.append(nn.Linear(width, 1)); self.network = nn.Sequential(*layers)
    def forward(self, inputs): return self.network(inputs).squeeze(-1)

class TransformerRegressor(nn.Module):
    def __init__(self, input_dim, sequence_length, d_model, heads, feedforward_dim):
        super().__init__(); self.embedding = nn.Linear(int(input_dim), d_model)
        position = torch.arange(int(sequence_length)).unsqueeze(1); divisor = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        encoding = torch.zeros(int(sequence_length), d_model); encoding[:, 0::2] = torch.sin(position * divisor); encoding[:, 1::2] = torch.cos(position * divisor)
        self.register_buffer('positional_encoding', encoding.unsqueeze(0)); self.register_buffer('no_future_mask', torch.triu(torch.ones(int(sequence_length), int(sequence_length), dtype=torch.bool), diagonal=1))
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=heads, dim_feedforward=feedforward_dim, dropout=.1, activation='gelu', batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=1); self.head = nn.Linear(d_model, 1)
    def forward(self, inputs): return self.head(self.encoder(self.embedding(inputs) + self.positional_encoding, mask=self.no_future_mask)[:, -1]).squeeze(-1)

def _neural_tensor(values, family, lag_order, parent_count):
    tensor = torch.tensor(np.asarray(values, dtype=np.float64), dtype=torch.float32)
    return tensor.reshape(-1, int(lag_order), int(parent_count)) if family == 'TRANSFORMER' else tensor

def _make_neural(family, specification):
    architecture, learning_rate = specification['configuration']
    width = len(flexible_feature_names(specification['parent_blocks'], specification['lag_order']))
    if family == 'MLP': network = MLPRegressor(width, architecture)
    else: network = TransformerRegressor(len(specification['parent_blocks']), specification['lag_order'], *architecture)
    return network.to(DEVICE), learning_rate

def _train_neural(family, specification, seed, calibration, validation=None, epochs=None):
    set_seed(seed); target = specification['target']; features = flexible_feature_names(specification['parent_blocks'], specification['lag_order']); scale = fit_state_scaler(calibration, target, specification['parent_blocks'], specification['lag_order'])
    train_x = _neural_tensor(_scaled_features(calibration, features, scale), family, specification['lag_order'], len(specification['parent_blocks'])); train_y = torch.tensor((calibration[target].to_numpy(dtype=np.float64) - scale['TARGET'][0]) / scale['TARGET'][1], dtype=torch.float32)
    network, learning_rate = _make_neural(family, specification); optimiser = torch.optim.AdamW(network.parameters(), lr=learning_rate, weight_decay=1e-4); loss = nn.MSELoss(); loader = DataLoader(TensorDataset(train_x, train_y), batch_size=CONFIG['batch_size'], shuffle=True)
    if epochs is None:
        if validation is None: raise ValueError('validation required for early stopping')
        valid_x = _neural_tensor(_scaled_features(validation, features, scale), family, specification['lag_order'], len(specification['parent_blocks'])).to(DEVICE); valid_y = torch.tensor((validation[target].to_numpy(dtype=np.float64) - scale['TARGET'][0]) / scale['TARGET'][1], dtype=torch.float32).to(DEVICE)
        best, state, best_epoch, wait = float('inf'), None, 0, 0
        for epoch in range(1, CONFIG['max_epochs'] + 1):
            network.train()
            for x, y in loader:
                optimiser.zero_grad(); value = loss(network(x.to(DEVICE)), y.to(DEVICE)); value.backward()
                if family == 'TRANSFORMER': torch.nn.utils.clip_grad_norm_(network.parameters(), max_norm=1.0)
                optimiser.step()
            network.eval()
            with torch.no_grad(): current = float(loss(network(valid_x), valid_y).item())
            if current < best - CONFIG['min_delta']: best, state, best_epoch, wait = current, {key: value.detach().cpu().clone() for key, value in network.state_dict().items()}, epoch, 0
            else: wait += 1
            if wait >= CONFIG['patience']: break
        network.load_state_dict(state); return network, scale, best_epoch
    for _ in range(int(epochs)):
        network.train()
        for x, y in loader:
            optimiser.zero_grad(); value = loss(network(x.to(DEVICE)), y.to(DEVICE)); value.backward()
            if family == 'TRANSFORMER': torch.nn.utils.clip_grad_norm_(network.parameters(), max_norm=1.0)
            optimiser.step()
    return network, scale, int(epochs)

def _predict_neural(networks, scales, frame, family, specification):
    if frame.empty: return np.array([], dtype=float)
    features = flexible_feature_names(specification['parent_blocks'], specification['lag_order']); predictions = []
    for network, scale in zip(networks, scales):
        inputs = _neural_tensor(_scaled_features(frame, features, scale), family, specification['lag_order'], len(specification['parent_blocks'])).to(DEVICE); network.eval()
        with torch.no_grad(): values = network(inputs).detach().cpu().numpy()
        predictions.append(values * scale['TARGET'][1] + scale['TARGET'][0])
    return np.mean(np.vstack(predictions), axis=0)


In [3]:
# Step-2 event-unaware forecasts, public return contract, and manual reference path.
PUBLIC_API_KEYS = {'structure', 'parametric_ordinary', 'parametric_eventless', 'flexible_spec', 'flexible_ordinary', 'flexible_eventless', 'event_mapping', 'event_weights', 'pooled_event_weights', 'ordinary_response_ratios', 'model_specification', 'diagnostics', 'run_metadata'}

def _metrics(actual, forecast):
    actual, forecast = np.asarray(actual, dtype=float), np.asarray(forecast, dtype=float)
    valid = np.isfinite(actual) & np.isfinite(forecast)
    error = actual[valid] - forecast[valid]
    return {'n': int(valid.sum()), 'RMSE': float(np.sqrt(np.mean(error ** 2))), 'MAE': float(np.mean(np.abs(error)))}

def frozen_n16_structure():
    source = authoritative_specification_row(PROCESSED / '16_model_specification.csv')
    retained = pd.DataFrame([
        {'source': 'Q', 'target': 'R', 'retained': str(source['retain_Q_to_R']).strip().lower() in {'true', '1'}},
        {'source': 'I', 'target': 'R', 'retained': str(source['retain_I_to_R']).strip().lower() in {'true', '1'}},
        {'source': 'Q', 'target': 'I', 'retained': str(source['retain_Q_to_I']).strip().lower() in {'true', '1'}},
        {'source': 'R', 'target': 'I', 'retained': str(source['retain_R_to_I']).strip().lower() in {'true', '1'}},
    ])
    return {'selected_orders': {'R': int(source['selected_realised_order']), 'I': int(source['selected_implied_order'])}, 'retained_structure': retained, 'metadata': {'source': 'FROZEN_N16_EXPORT'}}

def _ordinary_rows(frame, target, family, stage, prediction, event_days):
    out = frame[['model_day', 'sample_split', target]].copy().rename(columns={target: 'actual'})
    out['target_model_day'] = out['model_day']; out['target'] = target; out['model'] = family; out['forecast_stage'] = stage
    out['background_forecast'] = np.asarray(prediction, dtype=float)
    out['equation_eligible'] = True
    out['valid_positive_background'] = np.isfinite(out['background_forecast']) & out['background_forecast'].gt(0)
    out['ratio_defined'] = out['valid_positive_background']
    out['response_ratio'] = np.where(out['ratio_defined'], out['actual'] / out['background_forecast'], np.nan)
    out['known_event_target_flag'] = out['model_day'].isin(event_days)
    out['ordinary_flag'] = ~out['known_event_target_flag']; out['is_out_of_sample'] = True
    return out[['model_day', 'target_model_day', 'sample_split', 'forecast_stage', 'target', 'model', 'actual', 'background_forecast', 'response_ratio', 'valid_positive_background', 'ratio_defined', 'equation_eligible', 'known_event_target_flag', 'ordinary_flag', 'is_out_of_sample']]

def _legacy_estimate_flexible_ordinary_forecasts(model_panel, event_history, flexible_spec, config=CONFIG):
    panel = prepare_model_panel(model_panel, config)
    selection = validate_flexible_spec(flexible_spec)
    mapping = target_event_mapping(_normalise_events(event_history, panel, 'event_history'), panel)
    rows, fitted = [], {}
    for (target, family), item in sorted(selection['selected'].items()):
        features = flexible_feature_names(item['parent_blocks'], item['lag_order'])
        train = _model_frame(panel, target, 'train', item['parent_blocks'], item['lag_order'])
        validation = _model_frame(panel, target, 'validation', item['parent_blocks'], item['lag_order'])
        test = _model_frame(panel, target, 'test', item['parent_blocks'], item['lag_order'])
        if family in {'RF', 'GB'}:
            development = _tree(family, item).fit(train[features], train[target])
            validation_prediction = development.predict(validation[features])
            calibration = pd.concat([train, validation], ignore_index=True)
            final = _tree(family, item).fit(calibration[features], calibration[target])
            test_prediction = final.predict(test[features])
        else:
            development_models, development_scales = [], []
            for seed in config['neural_seeds']:
                network, scale, _ = _train_neural(family, item, seed, train, validation)
                development_models.append(network); development_scales.append(scale)
            validation_prediction = _predict_neural(development_models, development_scales, validation, family, item)
            final_models, final_scales, calibration = [], [], pd.concat([train, validation], ignore_index=True)
            for seed in config['neural_seeds']:
                network, scale, _ = _train_neural(family, item, seed, calibration, epochs=item['final_epochs'])
                final_models.append(network); final_scales.append(scale)
            final = (final_models, final_scales)
            test_prediction = _predict_neural(final_models, final_scales, test, family, item)
            development = (development_models, development_scales)
        fitted[(target, family, 'TRAIN_TO_VALIDATION')] = development
        fitted[(target, family, 'TRAIN_VALIDATION_TO_TEST')] = final
        event_days = set(mapping.loc[mapping['target'].eq(target), 'target_model_day'])
        rows.extend([_ordinary_rows(validation, target, family, 'TRAIN_TO_VALIDATION', validation_prediction, event_days), _ordinary_rows(test, target, family, 'TRAIN_VALIDATION_TO_TEST', test_prediction, event_days)])
    return {'forecasts': pd.concat(rows, ignore_index=True), 'models': fitted, 'metadata': {'event_mask_used': False, 'forecast_origin': 'FINAL_N19_STEP_2'}}

def assert_ratio_propagation(results):
    for frame, definition in [(results['event_weights'], 'weight_defined'), (results['ordinary_response_ratios'], 'ratio_defined')]:
        valid = frame[definition].astype(bool)
        assert np.allclose(frame.loc[valid, 'response_ratio'], frame.loc[valid, 'actual'] / frame.loc[valid, 'background_forecast'])
    return True

def _legacy_n19_equivalence_summary(results, tolerances=None):
    tolerances = {'tree': 1e-10, 'neural': 1e-6} | ({} if tolerances is None else tolerances)
    reference = pd.read_csv(PROCESSED / '19_ordinary_response_ratios.csv', parse_dates=['model_day', 'target_model_day'])
    actual = results['ordinary_response_ratios'].copy()
    keys = ['model_day', 'target', 'model', 'sample_split', 'forecast_stage']
    merged = actual.merge(reference, on=keys, suffixes=('_api', '_ref'), how='outer', indicator=True)
    common = merged.loc[merged['_merge'].eq('both')].copy()
    rows = []
    for family in MODEL_FAMILIES:
        part = common.loc[common['model'].eq(family)].copy(); tolerance = tolerances['neural'] if family in {'MLP', 'TRANSFORMER'} else tolerances['tree']
        background = np.abs(part['background_forecast_api'] - part['background_forecast_ref'])
        ratio = np.abs(part['response_ratio_api'] - part['response_ratio_ref'])
        status = part['valid_positive_background_api'].eq(part['valid_positive_background_ref']) & part['ratio_defined'].eq(part['valid_positive_background_ref'])
        valid = part['valid_positive_background_api'] & part['valid_positive_background_ref']
        identity = part.loc[valid, 'response_ratio_api'] - part.loc[valid, 'response_ratio_ref'] - part.loc[valid, 'actual_api'] * (1 / part.loc[valid, 'background_forecast_api'] - 1 / part.loc[valid, 'background_forecast_ref'])
        rows.append({'component': 'ordinary_' + family, 'n_compared': len(part), 'max_abs_background_difference': float(background.max()) if len(part) else np.nan, 'max_abs_ratio_difference': float(ratio.max()) if len(part) else np.nan, 'max_abs_ratio_identity_residual': float(np.abs(identity).max()) if len(identity) else np.nan, 'tolerance': tolerance, 'passed': bool(len(part) and background.max() <= tolerance and ratio.max() <= tolerance and status.all() and np.allclose(identity, 0.0, atol=tolerance, rtol=0.0))})
    return pd.DataFrame(rows)

def _legacy_estimate_event_weights(model_panel, event_history, evaluation_events=None, structure=None, flexible_spec=None, config=None):
    config = CONFIG if config is None else config
    structure_source = 'ESTIMATED_THIS_RUN' if structure is None else 'SUPPLIED'
    flexible_source = 'SELECTED_THIS_RUN' if flexible_spec is None else 'SUPPLIED'
    structure = estimate_dependence_structure(model_panel, config) if structure is None else structure
    flexible_spec = select_flexible_models(model_panel, event_history, config) if flexible_spec is None else validate_flexible_spec(flexible_spec)
    parametric_ordinary = fit_parametric_models(model_panel, structure, config)
    parametric_eventless = estimate_parametric_event_weights(model_panel, event_history, evaluation_events, structure, config)
    flexible_ordinary = estimate_flexible_ordinary_forecasts(model_panel, event_history, flexible_spec, config)
    flexible_eventless = estimate_flexible_event_weights(model_panel, event_history, evaluation_events, flexible_spec, config)
    event_weights = pd.concat([parametric_eventless['event_weights'], flexible_eventless['event_weights']], ignore_index=True)
    mapping = build_eventless_calibration_sample(model_panel, event_history, evaluation_events, config)['evaluation_mapping']
    run_metadata = pd.DataFrame([{'event_alignment_version': ALIGNMENT_VERSION, 'iv_pricing_hours_ny': '17:00-16:59', 'iv_daily_observation_interpretation': 'END_OF_BLOOMBERG_PRICING_DAY_STATE', 'bloomberg_pricing_session_identified': True, 'bloomberg_snapshot_time_identified': False, 'approximate_10am_iv_cut_used_for_production': False, 'linus_equation_5_used': False, 'structure_source': structure_source, 'flexible_spec_source': flexible_source, 'api_mode': 'FULL' if structure_source == 'ESTIMATED_THIS_RUN' and flexible_source == 'SELECTED_THIS_RUN' else 'REUSE_OR_PARTIAL_REUSE', 'run_full_rediscovery': bool(globals().get('RUN_FULL_REDISCOVERY', False)), 'test_used_for_selection': False}])
    return {'structure': structure, 'parametric_ordinary': parametric_ordinary, 'parametric_eventless': parametric_eventless, 'flexible_spec': flexible_spec, 'flexible_ordinary': flexible_ordinary, 'flexible_eventless': flexible_eventless, 'event_mapping': mapping, 'event_weights': event_weights, 'pooled_event_weights': pool_event_weights(event_weights, config), 'ordinary_response_ratios': flexible_ordinary['forecasts'], 'model_specification': _specification_table(flexible_spec['selected']), 'diagnostics': {'ratio_propagation_ready': True, 'ordinary_event_unaware': True}, 'run_metadata': run_metadata}

def _legacy_export_n20_results(results, output_dir=PROCESSED, equivalence_summary=None):
    assert_ratio_propagation(results)
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    ordinary = results['ordinary_response_ratios'].copy()
    summary = ordinary.groupby(['sample_split', 'target', 'model', 'forecast_stage'], as_index=False).agg(n_rows=('response_ratio', 'size'), n_ratio_defined=('ratio_defined', 'sum'), median_response_ratio=('response_ratio', 'median'))
    specification = pd.DataFrame([('event_alignment_version', ALIGNMENT_VERSION), ('iv_pricing_hours_ny', '17:00-16:59'), ('iv_daily_observation_interpretation', 'END_OF_BLOOMBERG_PRICING_DAY_STATE'), ('bloomberg_pricing_session_identified', 'True'), ('bloomberg_snapshot_time_identified', 'False'), ('approximate_10am_iv_cut_used_for_production', 'False'), ('linus_equation_5_used', 'False')], columns=['item', 'value']).assign(section='API')
    exports = {'20_api_reference_event_weights.csv': results['event_weights'], '20_api_event_mapping.csv': results['event_mapping'], '20_api_ordinary_response_ratios.csv': ordinary, '20_api_ordinary_response_ratio_summary.csv': summary, '20_api_run_metadata.csv': results['run_metadata'], '20_api_specification.csv': specification, '20_api_equivalence_summary.csv': pd.DataFrame() if equivalence_summary is None else equivalence_summary}
    for filename, frame in exports.items(): frame.to_csv(output_dir / filename, index=False)
    return exports


In [4]:
# Complete self-contained N20 callable implementations.
def _tree(family, item):
    return (RandomForestRegressor(**RF_FIXED, **item['configuration']) if family == 'RF' else GradientBoostingRegressor(**GB_FIXED, **item['configuration']))

def _rmse(actual, forecast):
    return float(np.sqrt(np.mean((np.asarray(actual, dtype=float) - np.asarray(forecast, dtype=float)) ** 2)))

def _prediction(fitted, family, item, frame):
    features = flexible_feature_names(item['parent_blocks'], item['lag_order'])
    if family in {'RF', 'GB'}: return fitted.predict(frame[features])
    return _predict_neural(fitted[0], fitted[1], frame, family, item)

def select_flexible_models(model_panel, event_history=None, config=CONFIG):
    panel = prepare_model_panel(model_panel, config); selected, records = {}, []
    for family in MODEL_FAMILIES:
        for target in TARGETS:
            common_train = _model_frame(panel, target, 'train', (target,), 1, common_support=True)
            common_valid = _model_frame(panel, target, 'validation', (target,), 1, common_support=True)
            candidates = []
            for parents in PARENT_SETS[target]:
                for lag_order in range(1, config['max_lag'] + 1):
                    item = {'target': target, 'model_family': family, 'parent_blocks': parents, 'lag_order': lag_order, 'config_id': PROBE_CONFIG[family], 'configuration': GRIDS[family][PROBE_CONFIG[family]]}
                    features = flexible_feature_names(parents, lag_order)
                    if family in {'RF', 'GB'}: score = _rmse(common_valid[target], _tree(family, item).fit(common_train[features], common_train[target]).predict(common_valid[features]))
                    else:
                        scores = []
                        for seed in config['neural_seeds']:
                            network, scale, _ = _train_neural(family, item, seed, common_train, common_valid)
                            scores.append(_rmse(common_valid[target], _predict_neural([network], [scale], common_valid, family, item)))
                        score = float(np.mean(scores))
                    candidates.append((score, len(parents) - 1, lag_order, parents))
                    records.append({'model': family, 'target': target, 'parent_blocks': '|'.join(parents), 'lag_order': lag_order, 'validation_RMSE': score})
            minimum = min(row[0] for row in candidates); _, _, lag_order, parents = sorted([row for row in candidates if row[0] <= config['parsimony_band'] * minimum], key=lambda row: (row[1], row[2], row[0], row[3]))[0]
            train = _model_frame(panel, target, 'train', parents, lag_order); valid = _model_frame(panel, target, 'validation', parents, lag_order); hyper = []
            for config_id, configuration in GRIDS[family].items():
                item = {'target': target, 'model_family': family, 'parent_blocks': parents, 'lag_order': lag_order, 'config_id': config_id, 'configuration': configuration}
                features = flexible_feature_names(parents, lag_order)
                if family in {'RF', 'GB'}: score, epoch = _rmse(valid[target], _tree(family, item).fit(train[features], train[target]).predict(valid[features])), np.nan
                else:
                    scores, epochs = [], []
                    for seed in config['neural_seeds']:
                        network, scale, this_epoch = _train_neural(family, item, seed, train, valid); scores.append(_rmse(valid[target], _predict_neural([network], [scale], valid, family, item))); epochs.append(this_epoch)
                    score, epoch = float(np.mean(scores)), int(np.median(epochs))
                hyper.append((score, config_id, epoch, item))
            _, _, epoch, item = sorted(hyper, key=lambda row: (row[0], row[1]))[0]
            if family in {'MLP', 'TRANSFORMER'}: item['final_epochs'] = epoch
            selected[(target, family)] = item
    checked = validate_flexible_spec({'selected': selected}); checked['structure_search_validation'] = pd.DataFrame(records); checked['model_specification'] = _specification_table(checked['selected']); checked['metadata'] = {'mode': 'FULL', 'structure_candidate_count': len(records), 'event_history_used_for_selection': False, 'test_used_for_selection': False}; assert len(records) == 384
    return checked

def _legacy_estimate_dependence_structure(model_panel, config=CONFIG):
    panel = prepare_model_panel(model_panel, config); orders = {}; rows = []
    for target in TARGETS:
        for lag in range(1, config.get('gc_max_lag', 20) + 1):
            columns = [target] + [f'{state}_lag{j}' for state in CONFIG['states'] for j in range(1, lag + 1)]
            frame = panel.loc[panel['sample_split'].eq('train') & np.isfinite(panel[columns].to_numpy(dtype=float)).all(axis=1)]
            rows.append({'target': target, 'history_order': lag, 'BIC': sm.OLS(frame[target], sm.add_constant(frame[columns[1:]], has_constant='add')).fit().bic})
        orders[target] = int(pd.DataFrame(rows).loc[lambda x: x.target.eq(target)].sort_values(['BIC', 'history_order']).iloc[0].history_order)
    frozen = frozen_n16_structure(); frozen['selected_orders'] = orders; frozen['order_scan'] = pd.DataFrame(rows); frozen['metadata'] = {'source': 'ESTIMATED_THIS_RUN'}; return frozen

def _legacy_fit_parametric_models(model_panel, structure, config=CONFIG):
    panel = prepare_model_panel(model_panel, config); pieces = []
    for target in TARGETS:
        features = _parametric_features(structure, target)
        train = panel.loc[panel['sample_split'].eq('train') & np.isfinite(panel[[target, *features]].to_numpy(dtype=float)).all(axis=1)]
        valid = panel.loc[panel['sample_split'].eq('validation') & np.isfinite(panel[[target, *features]].to_numpy(dtype=float)).all(axis=1)]
        test = panel.loc[panel['sample_split'].eq('test') & np.isfinite(panel[[target, *features]].to_numpy(dtype=float)).all(axis=1)]
        development = sm.OLS(train[target], sm.add_constant(train[features], has_constant='add')).fit(); final_frame = pd.concat([train, valid], ignore_index=True); final = sm.OLS(final_frame[target], sm.add_constant(final_frame[features], has_constant='add')).fit()
        for frame, fitted, stage in [(valid, development, 'TRAIN_TO_VALIDATION'), (test, final, 'TRAIN_VALIDATION_TO_TEST')]:
            pieces.append(pd.DataFrame({'model_day': frame['model_day'], 'target': target, 'sample_split': frame['sample_split'], 'forecast_stage': stage, 'actual': frame[target], 'background_forecast': fitted.predict(sm.add_constant(frame[features], has_constant='add'))}))
    forecasts = pd.concat(pieces, ignore_index=True); return {'forecasts': forecasts, 'metrics': forecasts.groupby(['target', 'sample_split', 'forecast_stage']).apply(lambda x: pd.Series(_metrics(x.actual, x.background_forecast)), include_groups=False).reset_index(), 'metadata': {'event_history_used': False}}

def _parametric_features(structure, target):
    order = int(structure['selected_orders'][target]); retained = structure['retained_structure'].set_index(['source', 'target'])['retained'].to_dict(); return [f'{state}_lag{lag}' for state in CONFIG['states'] if state == target or bool(retained.get((state, target), False)) for lag in range(1, order + 1)]

def _event_rows(events, panel, forecast, target, family, stage, oos, features):
    out = events.copy().merge(panel[['model_day', target, *features]], left_on='target_model_day', right_on='model_day', how='left', validate='many_to_one').drop(columns='model_day').rename(columns={target: 'actual'}).merge(forecast, left_on='target_model_day', right_on='model_day', how='left', validate='many_to_one').drop(columns='model_day')
    out['equation_eligible'] = np.isfinite(out[['actual', *features]].to_numpy(dtype=float)).all(axis=1); out['background_positive'] = out['equation_eligible'] & out['background_forecast'].gt(0); out['weight_defined'] = out['background_positive']; out['response_ratio'] = np.where(out['weight_defined'], out['actual'] / out['background_forecast'], np.nan); out['event_weight'] = out['response_ratio']; out['undefined_reason'] = np.where(out['weight_defined'], '', 'undefined'); out['model'] = family; out['forecast_stage'] = stage; out['is_out_of_sample'] = oos; return out

def _legacy_estimate_parametric_event_weights(model_panel, event_history, evaluation_events=None, structure=None, config=CONFIG):
    structure = frozen_n16_structure() if structure is None else structure; built = build_eventless_calibration_sample(model_panel, event_history, evaluation_events, config); panel, mapping = built['panel'], built['evaluation_mapping']; rows = []
    for target in TARGETS:
        features = _parametric_features(structure, target)
        for split, calibration_splits, stage, oos in [('validation', ('train',), 'EVENTLESS_TRAIN_TO_VALIDATION', True), ('test', ('train', 'validation'), 'EVENTLESS_TRAIN_VALIDATION_TO_TEST', True)]:
            calibration = panel.loc[panel['sample_split'].isin(calibration_splits) & ~panel['is_target_event_day'] & np.isfinite(panel[[target, *features]].to_numpy(dtype=float)).all(axis=1)]; fitted = sm.OLS(calibration[target], sm.add_constant(calibration[features], has_constant='add')).fit(); events = mapping.loc[(mapping.target.eq(target)) & (mapping.sample_split.eq(split))]; days = events[['target_model_day']].drop_duplicates().rename(columns={'target_model_day': 'model_day'}).merge(panel[['model_day', *features]], on='model_day', how='left'); days = days.loc[np.isfinite(days[features].to_numpy(dtype=float)).all(axis=1)]; forecast = pd.DataFrame({'model_day': days.model_day, 'background_forecast': fitted.predict(sm.add_constant(days[features], has_constant='add'))}); rows.append(_event_rows(events, panel, forecast, target, 'PARAMETRIC_N17', stage, oos, features))
    return {'event_weights': pd.concat(rows, ignore_index=True), 'metadata': {'eventless_parameters_refit': True}}

def _legacy_estimate_flexible_event_weights(model_panel, event_history, evaluation_events=None, flexible_spec=None, config=CONFIG):
    selection = select_flexible_models(model_panel, event_history, config) if flexible_spec is None else validate_flexible_spec(flexible_spec); built = build_eventless_calibration_sample(model_panel, event_history, evaluation_events, config); panel, mapping = built['panel'], built['evaluation_mapping']; rows, fitted = [], {}
    for (target, family), item in sorted(selection['selected'].items()):
        features = flexible_feature_names(item['parent_blocks'], item['lag_order'])
        for split, calibration_splits, stage, oos in [('validation', ('train',), 'EVENTLESS_TRAIN_TO_VALIDATION', True), ('test', ('train', 'validation'), 'EVENTLESS_TRAIN_VALIDATION_TO_TEST', True)]:
            calibration = panel.loc[panel['sample_split'].isin(calibration_splits) & ~panel['is_target_event_day'] & np.isfinite(panel[[target, *features]].to_numpy(dtype=float)).all(axis=1)]; events = mapping.loc[(mapping.target.eq(target)) & (mapping.sample_split.eq(split))]; days = events[['target_model_day']].drop_duplicates().rename(columns={'target_model_day': 'model_day'}).merge(panel[['model_day', *features]], on='model_day', how='left'); days = days.loc[np.isfinite(days[features].to_numpy(dtype=float)).all(axis=1)]
            if family in {'RF', 'GB'}: model = _tree(family, item).fit(calibration[features], calibration[target])
            else:
                networks, scales = [], []
                for seed in config['neural_seeds']:
                    network, scale, _ = _train_neural(family, item, seed, calibration, epochs=item['final_epochs']); networks.append(network); scales.append(scale)
                model = (networks, scales)
            fitted[(target, family, stage)] = model; forecast = pd.DataFrame({'model_day': days.model_day, 'background_forecast': _prediction(model, family, item, days)}); rows.append(_event_rows(events, panel, forecast, target, family, stage, oos, features))
    return {'event_weights': pd.concat(rows, ignore_index=True), 'models': fitted, 'metadata': {'eventless_parameters_refit': True, 'scalers_refit': True}}

def _legacy_pool_event_weights(event_weights, config=CONFIG):
    keys = ['event_label', 'sample_split', 'forecast_stage', 'is_out_of_sample', 'target', 'model']; rows = []; rng = np.random.default_rng(config.get('bootstrap_seed', 19))
    for key, group in event_weights.groupby(keys, dropna=False):
        values = group.loc[group.equation_eligible.astype(bool) & group.weight_defined.astype(bool), 'event_weight'].dropna().to_numpy(dtype=float); median = float(np.median(values)) if len(values) else np.nan; lower = upper = np.nan
        if len(values): draws = rng.choice(values, size=(int(config.get('bootstrap_reps', 10000)), len(values)), replace=True); lower, upper = np.quantile(np.median(draws, axis=1), [.025, .975])
        rows.append(dict(zip(keys, key)) | {'n_occurrences': len(group), 'n_valid': len(values), 'median_event_weight': median, 'bootstrap_ci_lower': lower, 'bootstrap_ci_upper': upper})
    return pd.DataFrame(rows)


In [5]:
# Final integration helpers: exact N17/N18 support, bounded N16 FULL discovery, and coverage-safe comparisons.
FULL_N16_MAX_LAG = 12

def _specification_table(selected):
    if isinstance(selected, dict) and 'selected' in selected: selected = selected['selected']
    rows = []
    for (target, model_family), item in sorted(selected.items()):
        rows.append({'target': target, 'model_family': model_family, 'parent_blocks': '|'.join(item['parent_blocks']), 'lag_order': int(item['lag_order']), 'config_id': item['config_id'], 'selected_spec_id': item['config_id'], 'hyperparameters': str(item['configuration']), 'final_epochs': item.get('final_epochs', np.nan)})
    table = pd.DataFrame(rows).sort_values(['target', 'model_family']).reset_index(drop=True)
    assert len(table) == 8 and not table.duplicated(['target', 'model_family']).any()
    return table

def canonical_reference_inputs():
    model_panel = pd.read_csv(PROCESSED / '16_empirical_analysis_panel.csv', parse_dates=['model_day'])
    mapping = pd.read_csv(PROCESSED / '18_target_specific_event_mapping.csv', parse_dates=['realised_model_day', 'implied_model_day', 'target_model_day', 'event_timestamp_utc'])
    events = mapping.loc[mapping['target'].eq('R'), ['event_id', 'event_label', 'event_timestamp_utc', 'realised_model_day']].drop_duplicates('event_id').copy()
    events['implied_model_day'] = events['realised_model_day']
    events = events.rename(columns={'event_timestamp_utc': 'event_timestamp'}).sort_values('event_id').reset_index(drop=True)
    evaluation_ids = set(mapping.loc[mapping['sample_split'].isin(['validation', 'test']), 'event_id'])
    evaluation_events = events.loc[events['event_id'].isin(evaluation_ids)].copy()
    assert not events['event_id'].duplicated().any() and events['implied_model_day'].eq(events['realised_model_day']).all()
    return model_panel, events, evaluation_events

def _parametric_eligibility(panel, target):
    if target == 'R': return np.isfinite(panel[['R', 'Q_lag1', 'R_lag1', 'I_lag1']].to_numpy(dtype=float)).all(axis=1)
    columns = ['I'] + [f'{state}_lag{lag}' for state in CONFIG['states'] for lag in range(1, FULL_N16_MAX_LAG + 1)]
    return np.isfinite(panel[columns].to_numpy(dtype=float)).all(axis=1)

def _holm(frame, alpha):
    from statsmodels.stats.multitest import multipletests
    out = frame.copy(); reject, adjusted, _, _ = multipletests(out['raw_p_value'], alpha=alpha, method='holm')
    out['holm_adjusted_p_value'] = adjusted; out['holm_reject'] = reject; return out

def _conditional_gc(panel, target, source, lag_order, split):
    columns = [target] + [f'{state}_lag{lag}' for state in CONFIG['states'] for lag in range(1, lag_order + 1)]
    frame = panel.loc[panel['sample_split'].eq(split) & np.isfinite(panel[columns].to_numpy(dtype=float)).all(axis=1)]
    unrestricted = sm.OLS(frame[target], sm.add_constant(frame[columns[1:]], has_constant='add')).fit()
    restricted_features = [column for column in columns[1:] if not column.startswith(source + '_lag')]
    restricted = sm.OLS(frame[target], sm.add_constant(frame[restricted_features], has_constant='add')).fit()
    statistic, p_value, degrees = unrestricted.compare_f_test(restricted)
    return {'source': source, 'target': target, 'split': split, 'lag_order': lag_order, 'f_statistic': float(statistic), 'raw_p_value': float(p_value), 'df_diff': float(degrees), 'n_observations': len(frame)}

def _tdmi_diagnostics(panel, source, target, lag_order, config):
    frame = panel.loc[panel['sample_split'].eq('train'), [source, target]].dropna().reset_index(drop=True)
    rows = []; bins = int(config.get('tdmi_bins', 10)); shifts = int(config.get('tdmi_surrogates', 1000)); rng = np.random.default_rng(config.get('gc_random_seed', 16017))
    for lag in range(1, lag_order + 1):
        x, y = frame[source].iloc[:-lag].to_numpy(), frame[target].iloc[lag:].to_numpy(); edges_x = np.histogram_bin_edges(x, bins=bins); edges_y = np.histogram_bin_edges(y, bins=bins); observed = mutual_info_score(np.digitize(x, edges_x[1:-1]), np.digitize(y, edges_y[1:-1])); surrogate = [mutual_info_score(np.digitize(x, edges_x[1:-1]), np.digitize(np.roll(y, int(rng.integers(1, len(y)))), edges_y[1:-1])) for _ in range(shifts)]; rows.append({'source': source, 'target': target, 'lag': lag, 'tdmi': float(observed), 'raw_p_value': float((1 + np.sum(np.asarray(surrogate) >= observed)) / (shifts + 1)), 'n_pairs': len(x)})
    return _holm(pd.DataFrame(rows), config.get('gc_alpha', .05))

def estimate_dependence_structure(model_panel, config=CONFIG):
    panel = prepare_model_panel(model_panel, config); alpha = config.get('gc_alpha', .05); order_rows = []; selected_orders = {}
    for target in TARGETS:
        for lag_order in range(1, FULL_N16_MAX_LAG + 1):
            columns = [target] + [f'{state}_lag{lag}' for state in CONFIG['states'] for lag in range(1, lag_order + 1)]
            frame = panel.loc[panel['sample_split'].eq('train') & np.isfinite(panel[columns].to_numpy(dtype=float)).all(axis=1)]
            fit = sm.OLS(frame[target], sm.add_constant(frame[columns[1:]], has_constant='add')).fit(); order_rows.append({'target': target, 'history_order': lag_order, 'AIC': fit.aic, 'BIC': fit.bic, 'n_observations': len(frame)})
        selected_orders[target] = int(pd.DataFrame(order_rows).loc[lambda x: x.target.eq(target)].sort_values(['BIC', 'history_order']).iloc[0].history_order)
    edge_rows = []
    for target, source in [('R', 'Q'), ('R', 'I'), ('I', 'Q'), ('I', 'R')]:
        for split in ('train', 'validation'): edge_rows.append(_conditional_gc(panel, target, source, selected_orders[target], split))
    gc = pd.DataFrame(edge_rows)
    for split in ('train', 'validation'):
        index = gc['split'].eq(split); gc.loc[index, ['holm_adjusted_p_value', 'holm_reject']] = _holm(gc.loc[index], alpha)[['holm_adjusted_p_value', 'holm_reject']].to_numpy()
    retained = gc.groupby(['source', 'target'], as_index=False).agg(retained=('holm_reject', 'all'), train_raw_p=('raw_p_value', 'first'), validation_raw_p=('raw_p_value', 'last'), train_holm_p=('holm_adjusted_p_value', 'first'), validation_holm_p=('holm_adjusted_p_value', 'last'))
    tdmi = pd.concat([_tdmi_diagnostics(panel, source, target, FULL_N16_MAX_LAG, config) for source, target in [('Q', 'R'), ('I', 'R'), ('Q', 'I'), ('R', 'I')]], ignore_index=True)
    return {'selected_orders': selected_orders, 'retained_structure': retained, 'gc_results': gc, 'tdmi_results': tdmi, 'order_scan': pd.DataFrame(order_rows), 'metadata': {'source': 'ESTIMATED_THIS_RUN', 'maximum_lag': FULL_N16_MAX_LAG, 'test_used_for_selection': False, 'holm_correction': True, 'tdmi_train_only': True}}

def _legacy_full_fit_parametric_models(model_panel, structure, config=CONFIG):
    panel = prepare_model_panel(model_panel, config); pieces = []
    for target in TARGETS:
        features = _parametric_features(structure, target); eligibility = _parametric_eligibility(panel, target)
        train = panel.loc[panel['sample_split'].eq('train') & eligibility]; validation = panel.loc[panel['sample_split'].eq('validation') & eligibility]; test = panel.loc[panel['sample_split'].eq('test') & eligibility]
        development = sm.OLS(train[target], sm.add_constant(train[features], has_constant='add')).fit(); calibration = pd.concat([train, validation], ignore_index=True); final = sm.OLS(calibration[target], sm.add_constant(calibration[features], has_constant='add')).fit()
        for frame, fitted, stage in [(validation, development, 'TRAIN_TO_VALIDATION'), (test, final, 'TRAIN_VALIDATION_TO_TEST')]: pieces.append(pd.DataFrame({'model_day': frame['model_day'], 'target': target, 'sample_split': frame['sample_split'], 'forecast_stage': stage, 'actual': frame[target], 'background_forecast': fitted.predict(sm.add_constant(frame[features], has_constant='add'))}))
    forecasts = pd.concat(pieces, ignore_index=True); return {'forecasts': forecasts, 'metrics': forecasts.groupby(['target', 'sample_split', 'forecast_stage']).apply(lambda x: pd.Series(_metrics(x.actual, x.background_forecast)), include_groups=False).reset_index(), 'metadata': {'event_history_used': False, 'canonical_support': True}}

def _legacy_full_estimate_parametric_event_weights(model_panel, event_history, evaluation_events=None, structure=None, config=CONFIG):
    structure = frozen_n16_structure() if structure is None else structure; built = build_eventless_calibration_sample(model_panel, event_history, evaluation_events, config); panel, mapping = built['panel'], built['evaluation_mapping']; rows = []
    for target in TARGETS:
        features = _parametric_features(structure, target); eligibility = _parametric_eligibility(panel, target)
        for split, calibration_splits, stage, oos in [('validation', ('train',), 'EVENTLESS_TRAIN_TO_VALIDATION', True), ('test', ('train', 'validation'), 'EVENTLESS_TRAIN_VALIDATION_TO_TEST', True)]:
            calibration = panel.loc[panel['sample_split'].isin(calibration_splits) & ~panel['is_target_event_day'] & eligibility]; fitted = sm.OLS(calibration[target], sm.add_constant(calibration[features], has_constant='add')).fit(); events = mapping.loc[(mapping.target.eq(target)) & (mapping.sample_split.eq(split))]; days = events[['target_model_day']].drop_duplicates().rename(columns={'target_model_day': 'model_day'}).merge(panel[['model_day', *features]], on='model_day', how='left'); days = days.loc[np.isfinite(days[features].to_numpy(dtype=float)).all(axis=1)]; forecast = pd.DataFrame({'model_day': days.model_day, 'background_forecast': fitted.predict(sm.add_constant(days[features], has_constant='add'))}); rows.append(_event_rows(events, panel, forecast, target, 'PARAMETRIC_N17', stage, oos, features))
    return {'event_weights': pd.concat(rows, ignore_index=True), 'metadata': {'eventless_parameters_refit': True, 'canonical_support': True}}

def _coverage_row(component, actual, reference, keys, numeric, tolerance):
    merged = actual.merge(reference, on=keys, how='outer', suffixes=('_api', '_ref'), indicator=True); common = merged.loc[merged['_merge'].eq('both')]; left, right = int(merged['_merge'].eq('left_only').sum()), int(merged['_merge'].eq('right_only').sum()); difference = np.abs(common[numeric + '_api'] - common[numeric + '_ref']); return {'component': component, 'n_compared': len(common), 'n_left_only': left, 'n_right_only': right, 'key_coverage_passed': left == 0 and right == 0, 'max_abs_difference': float(difference.max()) if len(difference) else np.nan, 'tolerance': tolerance, 'passed': bool(len(common) and left == 0 and right == 0 and difference.max() <= tolerance)}

def _legacy_n19_equivalence_summary_v2(results, tolerances=None):
    tolerances = {'tree': 1e-10, 'neural': 1e-6} | ({} if tolerances is None else tolerances); rows = []
    ordinary_ref = pd.read_csv(PROCESSED / '19_ordinary_response_ratios.csv', parse_dates=['model_day', 'target_model_day']); ordinary = results['ordinary_response_ratios']; ordinary_keys = ['model_day', 'target', 'model', 'sample_split', 'forecast_stage']
    for family in MODEL_FAMILIES: rows.append(_coverage_row('ordinary_background_' + family, ordinary.loc[ordinary.model.eq(family)], ordinary_ref.loc[ordinary_ref.model.eq(family)], ordinary_keys, 'background_forecast', tolerances['neural'] if family in {'MLP', 'TRANSFORMER'} else tolerances['tree']))
    events_ref = pd.read_csv(PROCESSED / '19_event_response_weights.csv', parse_dates=['target_model_day']); event_keys = ['event_id', 'target_model_day', 'target', 'model', 'event_label']; rows.append(_coverage_row('event_backgrounds', results['event_weights'].loc[lambda x: x.model.isin(MODEL_FAMILIES)], events_ref.loc[lambda x: x.model.isin(MODEL_FAMILIES)], event_keys, 'background_forecast', tolerances['neural']))
    spec_ref = pd.read_csv(PROCESSED / '19_selected_model_specifications.csv').rename(columns={'model': 'model_family', 'selected_parent_blocks': 'parent_blocks', 'selected_lag_order': 'lag_order', 'hyperparameter_candidate_id': 'config_id'}); spec = results['model_specification']; spec_keys = ['target', 'model_family']; merged = spec.merge(spec_ref[spec_keys + ['parent_blocks', 'lag_order', 'config_id']], on=spec_keys, how='outer', suffixes=('_api', '_ref'), indicator=True); coverage = merged['_merge'].eq('both'); match = coverage & merged['parent_blocks_api'].eq(merged['parent_blocks_ref']) & merged['lag_order_api'].eq(merged['lag_order_ref']) & merged['config_id_api'].eq(merged['config_id_ref']); rows.append({'component': 'specifications', 'n_compared': int(coverage.sum()), 'n_left_only': int(merged['_merge'].eq('left_only').sum()), 'n_right_only': int(merged['_merge'].eq('right_only').sum()), 'key_coverage_passed': bool(merged['_merge'].eq('both').all()), 'max_abs_difference': 0.0 if match.all() else np.nan, 'tolerance': 0.0, 'passed': bool(match.all() and merged['_merge'].eq('both').all())})
    return pd.DataFrame(rows)


def _legacy_n19_equivalence_summary_final(results, tolerances=None):
    tolerances = {'tree': 1e-10, 'neural': 1e-6} | ({} if tolerances is None else tolerances); rows = []
    ordinary_ref = pd.read_csv(PROCESSED / '19_ordinary_response_ratios.csv', parse_dates=['model_day', 'target_model_day']); ordinary = results['ordinary_response_ratios']; ordinary_keys = ['model_day', 'target', 'model', 'sample_split', 'forecast_stage']
    for family in MODEL_FAMILIES:
        tolerance = tolerances['neural'] if family in {'MLP', 'TRANSFORMER'} else tolerances['tree']; merged = ordinary.loc[ordinary.model.eq(family)].merge(ordinary_ref.loc[ordinary_ref.model.eq(family)], on=ordinary_keys, how='outer', suffixes=('_api', '_ref'), indicator=True); common = merged.loc[merged['_merge'].eq('both')]; left, right = int(merged['_merge'].eq('left_only').sum()), int(merged['_merge'].eq('right_only').sum()); background = np.abs(common['background_forecast_api'] - common['background_forecast_ref']); ratio = np.abs(common['response_ratio_api'] - common['response_ratio_ref']); status = common['valid_positive_background_api'].eq(common['valid_positive_background_ref']) & common['ratio_defined'].eq(common['valid_positive_background_ref']); valid = common['valid_positive_background_api'] & common['valid_positive_background_ref']; identity = common.loc[valid, 'response_ratio_api'] - common.loc[valid, 'response_ratio_ref'] - common.loc[valid, 'actual_api'] * (1 / common.loc[valid, 'background_forecast_api'] - 1 / common.loc[valid, 'background_forecast_ref']); passed = bool(len(common) and left == 0 and right == 0 and background.max() <= tolerance and ratio.max() <= tolerance and status.all() and np.allclose(identity, 0.0, atol=tolerance, rtol=0.0)); rows.append({'component': 'ordinary_' + family, 'n_compared': len(common), 'n_left_only': left, 'n_right_only': right, 'key_coverage_passed': left == 0 and right == 0, 'max_abs_background_difference': float(background.max()) if len(background) else np.nan, 'max_abs_ratio_difference': float(ratio.max()) if len(ratio) else np.nan, 'max_abs_ratio_identity_residual': float(np.abs(identity).max()) if len(identity) else np.nan, 'tolerance': tolerance, 'passed': passed})
    events_ref = pd.read_csv(PROCESSED / '19_event_response_weights.csv', parse_dates=['target_model_day']); event_keys = ['event_id', 'target_model_day', 'target', 'model', 'event_label']; event_merge = results['event_weights'].loc[lambda x: x.model.isin(MODEL_FAMILIES)].merge(events_ref.loc[lambda x: x.model.isin(MODEL_FAMILIES)], on=event_keys, how='outer', suffixes=('_api', '_ref'), indicator=True); event_common = event_merge.loc[event_merge['_merge'].eq('both')]; event_difference = np.abs(event_common['background_forecast_api'] - event_common['background_forecast_ref']); rows.append({'component': 'event_weights', 'n_compared': len(event_common), 'n_left_only': int(event_merge['_merge'].eq('left_only').sum()), 'n_right_only': int(event_merge['_merge'].eq('right_only').sum()), 'key_coverage_passed': bool(event_merge['_merge'].eq('both').all()), 'max_abs_background_difference': float(event_difference.max()) if len(event_difference) else np.nan, 'tolerance': tolerances['neural'], 'passed': bool(len(event_common) and event_merge['_merge'].eq('both').all() and event_difference.max() <= tolerances['neural'])})
    spec_ref = pd.read_csv(PROCESSED / '19_selected_model_specifications.csv').rename(columns={'model': 'model_family', 'selected_parent_blocks': 'parent_blocks', 'selected_lag_order': 'lag_order', 'hyperparameter_candidate_id': 'config_id'}); spec = results['model_specification']; spec_keys = ['target', 'model_family']; spec_merge = spec.merge(spec_ref[spec_keys + ['parent_blocks', 'lag_order', 'config_id']], on=spec_keys, how='outer', suffixes=('_api', '_ref'), indicator=True); comparable = spec_merge['_merge'].eq('both'); matches = comparable & spec_merge['parent_blocks_api'].eq(spec_merge['parent_blocks_ref']) & spec_merge['lag_order_api'].eq(spec_merge['lag_order_ref']) & spec_merge['config_id_api'].eq(spec_merge['config_id_ref']); rows.append({'component': 'specifications', 'n_compared': int(comparable.sum()), 'n_left_only': int(spec_merge['_merge'].eq('left_only').sum()), 'n_right_only': int(spec_merge['_merge'].eq('right_only').sum()), 'key_coverage_passed': bool(spec_merge['_merge'].eq('both').all()), 'max_abs_difference': 0.0 if matches.all() else np.nan, 'tolerance': 0.0, 'passed': bool(matches.all() and spec_merge['_merge'].eq('both').all())})
    return pd.DataFrame(rows)


In [6]:

# Authoritative Exercise 5 API.  The legacy helpers above are retained only as
# private historical reference paths; all public forecasting and weighting calls
# below use the common forecaster contract.

from dataclasses import dataclass

RUN_MODE = "REUSE"
assert RUN_MODE == "REUSE"

@dataclass(frozen=True)
class ModelSpec:
    model_family: str
    target: str
    parent_blocks: tuple
    lag_order: int
    config_id: str | None = None
    configuration: object = None
    final_epochs: int | None = None
    parametric_features: tuple | None = None

    @property
    def feature_names(self):
        if self.parametric_features is not None:
            return list(self.parametric_features)
        return flexible_feature_names(self.parent_blocks, self.lag_order)

class BaseForecaster:
    """Small common fit/predict contract for every N20 forecasting family."""

    def __init__(self, spec, config=CONFIG):
        self.spec = spec
        self.config = config
        self.fitted = False

    @property
    def features(self):
        return self.spec.feature_names

    def fit(self, calibration, validation=None, fixed_epochs=None):
        raise NotImplementedError

    def predict(self, evaluation):
        raise NotImplementedError

class ParametricForecaster(BaseForecaster):
    def fit(self, calibration, validation=None, fixed_epochs=None):
        self.model = sm.OLS(
            calibration[self.spec.target],
            sm.add_constant(calibration[self.features], has_constant="add"),
        ).fit()
        self.fitted = True
        return self

    def predict(self, evaluation):
        if not self.fitted:
            raise RuntimeError("fit must be called before predict")
        return np.asarray(
            self.model.predict(
                sm.add_constant(evaluation[self.features], has_constant="add")
            ),
            dtype=float,
        )

class TreeForecaster(BaseForecaster):
    def fit(self, calibration, validation=None, fixed_epochs=None):
        item = {
            "configuration": self.spec.configuration,
            "model_family": self.spec.model_family,
        }
        self.model = _tree(self.spec.model_family, item)
        self.model.fit(calibration[self.features], calibration[self.spec.target])
        self.fitted = True
        return self

    def predict(self, evaluation):
        if not self.fitted:
            raise RuntimeError("fit must be called before predict")
        return np.asarray(self.model.predict(evaluation[self.features]), dtype=float)

class NeuralForecaster(BaseForecaster):
    def fit(self, calibration, validation=None, fixed_epochs=None):
        item = {
            "target": self.spec.target,
            "parent_blocks": self.spec.parent_blocks,
            "lag_order": self.spec.lag_order,
            "configuration": self.spec.configuration,
        }
        # Development fits use validation early stopping; final/eventless fits pass frozen epochs explicitly.
        epochs = fixed_epochs
        if epochs is None and validation is None:
            raise ValueError("neural development fitting requires validation rows")
        self.networks, self.scales = [], []
        for seed in self.config["neural_seeds"]:
            network, scale, _ = _train_neural(
                self.spec.model_family,
                item,
                seed,
                calibration,
                validation=validation,
                epochs=epochs,
            )
            self.networks.append(network)
            self.scales.append(scale)
        self.fitted = True
        return self

    def predict(self, evaluation):
        if not self.fitted:
            raise RuntimeError("fit must be called before predict")
        item = {
            "target": self.spec.target,
            "parent_blocks": self.spec.parent_blocks,
            "lag_order": self.spec.lag_order,
            "configuration": self.spec.configuration,
        }
        return _predict_neural(
            self.networks,
            self.scales,
            evaluation,
            self.spec.model_family,
            item,
        )

def build_forecaster(model_spec, config=CONFIG):
    """The only model-family dispatch point used by the generic workflow."""
    if model_spec.model_family == "PARAMETRIC_N17":
        return ParametricForecaster(model_spec, config)
    if model_spec.model_family in {"RF", "GB"}:
        return TreeForecaster(model_spec, config)
    if model_spec.model_family in {"MLP", "TRANSFORMER"}:
        return NeuralForecaster(model_spec, config)
    raise ValueError(f"Unsupported model family: {model_spec.model_family}")

def _flexible_model_specs(flexible_spec):
    selected = validate_flexible_spec(flexible_spec)["selected"]
    return [
        ModelSpec(
            model_family=family,
            target=target,
            parent_blocks=tuple(item["parent_blocks"]),
            lag_order=int(item["lag_order"]),
            config_id=item["config_id"],
            configuration=item["configuration"],
            final_epochs=item.get("final_epochs"),
        )
        for (target, family), item in sorted(selected.items())
    ]

def _parametric_model_specs(structure):
    return [
        ModelSpec(
            model_family="PARAMETRIC_N17",
            target=target,
            parent_blocks=tuple(),
            lag_order=int(structure["selected_orders"][target]),
            parametric_features=tuple(_parametric_features(structure, target)),
        )
        for target in TARGETS
    ]

def _usable_frame(panel, spec, splits):
    columns = [spec.target, *spec.feature_names]
    finite = np.isfinite(panel[columns].to_numpy(dtype=float)).all(axis=1)
    return panel.loc[panel["sample_split"].isin(splits) & finite].copy()

def fit_background(forecaster, calibration_rows, validation_rows=None, fixed_epochs=None):
    """Model-agnostic calibration operation used for ordinary and eventless fits."""
    return forecaster.fit(
        calibration_rows,
        validation=validation_rows,
        fixed_epochs=fixed_epochs,
    )

def calculate_background(forecaster, evaluation_rows):
    """Model-agnostic background calculation; the forecaster owns all internals."""
    return forecaster.predict(evaluation_rows)

def calculate_event_weights(actual, eventless_background):
    actual = np.asarray(actual, dtype=float)
    background = np.asarray(eventless_background, dtype=float)
    defined = np.isfinite(actual) & np.isfinite(background) & (background > 0)
    return np.where(defined, actual / background, np.nan), defined

def _ordinary_forecast_rows(frame, spec, stage, prediction, target_event_days):
    rows = _ordinary_rows(
        frame,
        spec.target,
        spec.model_family,
        stage,
        prediction,
        target_event_days,
    )
    rows["ordinary_background_forecast"] = rows["background_forecast"]
    rows["ordinary_response_ratio"] = rows["response_ratio"]
    return rows

def _ordinary_backgrounds(panel, specs, event_history, config):
    mapping = target_event_mapping(event_history, panel)
    rows, fitted = [], {}
    stage_plan = (
        ("validation", ("train",), "TRAIN_TO_VALIDATION", False),
        ("test", ("train", "validation"), "TRAIN_VALIDATION_TO_TEST", True),
    )
    for spec in specs:
        train = _usable_frame(panel, spec, ("train",))
        validation = _usable_frame(panel, spec, ("validation",))
        test = _usable_frame(panel, spec, ("test",))
        event_days = set(
            mapping.loc[mapping["target"].eq(spec.target), "target_model_day"]
        )
        for split, calibration_splits, stage, fixed in stage_plan:
            evaluation = validation if split == "validation" else test
            calibration = _usable_frame(panel, spec, calibration_splits)
            forecaster = build_forecaster(spec, config)
            fit_background(
                forecaster,
                calibration,
                validation_rows=evaluation if not fixed else None,
                fixed_epochs=spec.final_epochs if fixed else None,
            )
            prediction = calculate_background(forecaster, evaluation)
            rows.append(
                _ordinary_forecast_rows(
                    evaluation, spec, stage, prediction, event_days
                )
            )
            fitted[(spec.target, spec.model_family, stage)] = forecaster
    return {"forecasts": pd.concat(rows, ignore_index=True), "models": fitted}

def _eventless_backgrounds(panel, history_mapping, evaluation_mapping, specs, config):
    rows, fitted = [], {}
    stage_plan = (
        ("validation", ("train",), "EVENTLESS_TRAIN_TO_VALIDATION"),
        ("test", ("train", "validation"), "EVENTLESS_TRAIN_VALIDATION_TO_TEST"),
    )
    for spec in specs:
        # event_history controls calibration masking; evaluation_events controls returned rows only.
        target_history = history_mapping.loc[history_mapping["target"].eq(spec.target)]
        target_events = evaluation_mapping.loc[evaluation_mapping["target"].eq(spec.target)]
        masked_days = set(target_history["target_model_day"])
        for split, calibration_splits, stage in stage_plan:
            calibration = _usable_frame(panel, spec, calibration_splits)
            calibration = calibration.loc[~calibration["model_day"].isin(masked_days)]
            events = target_events.loc[target_events["sample_split"].eq(split)].copy()
            days = events[["target_model_day"]].drop_duplicates().rename(
                columns={"target_model_day": "model_day"}
            )
            days = days.merge(
                panel[["model_day", *spec.feature_names]],
                on="model_day",
                how="left",
                validate="one_to_one",
            )
            days = days.loc[
                np.isfinite(days[spec.feature_names].to_numpy(dtype=float)).all(axis=1)
            ]
            forecaster = build_forecaster(spec, config)
            # N19's eventless neural path uses frozen final epochs on both stages.
            fit_background(
                forecaster,
                calibration,
                fixed_epochs=spec.final_epochs,
            )
            forecast = pd.DataFrame(
                {
                    "model_day": days["model_day"],
                    "background_forecast": calculate_background(forecaster, days),
                }
            )
            event_rows = _event_rows(
                events,
                panel,
                forecast,
                spec.target,
                spec.model_family,
                stage,
                True,
                spec.feature_names,
            )
            event_rows["eventless_background_forecast"] = event_rows[
                "background_forecast"
            ]
            event_rows["event_weight"], defined = calculate_event_weights(
                event_rows["actual"],
                event_rows["eventless_background_forecast"],
            )
            event_rows["response_ratio"] = event_rows["event_weight"]
            event_rows["weight_defined"] = defined
            event_rows["background_positive"] = defined
            event_rows["undefined_reason"] = np.where(
                defined,
                "",
                "nonpositive_or_nonfinite_eventless_background",
            )
            rows.append(event_rows)
            fitted[(spec.target, spec.model_family, stage)] = forecaster
    return {"event_weights": pd.concat(rows, ignore_index=True), "models": fitted}

def fit_parametric_models(model_panel, structure, config=CONFIG):
    panel = prepare_model_panel(model_panel, config)
    specs = _parametric_model_specs(structure)
    history = pd.DataFrame(
        columns=["event_id", "event_label", "realised_model_day"]
    )
    # Ordinary forecasts do not mask events; an empty history simply gives flags.
    ordinary = _ordinary_backgrounds(
        panel,
        specs,
        history,
        config,
    )
    forecasts = ordinary["forecasts"]
    return {
        "forecasts": forecasts,
        "metrics": forecasts.groupby(
            ["target", "sample_split", "forecast_stage"],
            as_index=False,
        ).apply(
            lambda group: pd.Series(
                _metrics(group["actual"], group["background_forecast"])
            ),
            include_groups=False,
        ),
        "models": ordinary["models"],
        "metadata": {"event_history_used": False, "common_interface": True},
    }

def estimate_parametric_event_weights(
    model_panel,
    event_history,
    evaluation_events=None,
    structure=None,
    config=CONFIG,
):
    structure = frozen_n16_structure() if structure is None else structure
    built = build_eventless_calibration_sample(
        model_panel, event_history, evaluation_events, config
    )
    backgrounds = _eventless_backgrounds(
        built["panel"],
        built["event_mapping"],
        built["evaluation_mapping"],
        _parametric_model_specs(structure),
        config,
    )
    return {
        **backgrounds,
        "metadata": {
            "eventless_parameters_refit": True,
            "scalers_refit": False,
            "common_interface": True,
        },
    }

def estimate_flexible_ordinary_forecasts(
    model_panel,
    event_history,
    flexible_spec,
    config=CONFIG,
):
    panel = prepare_model_panel(model_panel, config)
    history = _normalise_events(event_history, panel, "event_history")
    return {
        **_ordinary_backgrounds(
            panel,
            _flexible_model_specs(flexible_spec),
            history,
            config,
        ),
        "metadata": {
            "event_mask_used": False,
            "forecast_origin": "FROZEN_N19_REUSE",
            "common_interface": True,
        },
    }

def estimate_flexible_event_weights(
    model_panel,
    event_history,
    evaluation_events=None,
    flexible_spec=None,
    config=CONFIG,
):
    flexible_spec = (
        frozen_n19_flexible_spec() if flexible_spec is None else flexible_spec
    )
    built = build_eventless_calibration_sample(
        model_panel, event_history, evaluation_events, config
    )
    return {
        **_eventless_backgrounds(
            built["panel"],
            built["event_mapping"],
            built["evaluation_mapping"],
            _flexible_model_specs(flexible_spec),
            config,
        ),
        "metadata": {
            "eventless_parameters_refit": True,
            "scalers_refit": True,
            "common_interface": True,
        },
    }

def pool_event_weights(event_weights, config=CONFIG):
    """Final N18 clean-TEST median pool with N18 bootstrap provenance."""
    required = {"target", "model", "event_weight", "weight_defined"}
    if not required.issubset(event_weights.columns):
        raise ValueError("event weights lack fields required for pooling")
    pool = event_weights.copy()
    if "sample_split" in pool:
        pool = pool.loc[pool["sample_split"].eq("test")]
    if "is_clean_single_family" in pool:
        pool = pool.loc[pool["is_clean_single_family"].eq(True)]
    family_column = "family" if "family" in pool.columns else "event_label"
    rows = []
    for key, group in pool.groupby([family_column, "target", "model"], dropna=False):
        values = group.loc[group["weight_defined"].eq(True), "event_weight"].dropna().to_numpy(dtype=float)
        bootstrap_medians = np.array([], dtype=float)
        if len(values):
            rng = np.random.default_rng(int(config["bootstrap_seed"]))
            draws = rng.choice(
                values,
                size=(int(config["bootstrap_reps"]), len(values)),
                replace=True,
            )
            bootstrap_medians = np.median(draws, axis=1)
        rows.append(
            {
                "pool_specification": "PRIMARY_CLEAN_TEST",
                "family": key[0],
                "target": key[1],
                "model": key[2],
                "n_clean_occurrences": int(len(group)),
                "n_event_occurrences": int(len(group)),
                "n_equation_eligible": int(group["equation_eligible"].eq(True).sum()),
                "n_valid_positive_denominator": int(len(values)),
                "valid_fraction": float(len(values) / len(group)) if len(group) else np.nan,
                "eligible_positive_fraction": float(
                    len(values) / group["equation_eligible"].eq(True).sum()
                ) if group["equation_eligible"].eq(True).any() else np.nan,
                "observed_min": float(np.min(values)) if len(values) else np.nan,
                "observed_max": float(np.max(values)) if len(values) else np.nan,
                "q25": float(np.quantile(values, .25)) if len(values) else np.nan,
                "median": float(np.median(values)) if len(values) else np.nan,
                "q75": float(np.quantile(values, .75)) if len(values) else np.nan,
                "iqr": float(np.quantile(values, .75) - np.quantile(values, .25)) if len(values) else np.nan,
                "mean_secondary": float(np.mean(values)) if len(values) else np.nan,
                "mean": float(np.mean(values)) if len(values) else np.nan,
                "sample_std": float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
                "bootstrap_median_se": float(np.std(bootstrap_medians, ddof=1)) if len(bootstrap_medians) > 1 else np.nan,
                "bootstrap_ci_lower": float(np.quantile(bootstrap_medians, .025)) if len(bootstrap_medians) else np.nan,
                "bootstrap_ci_upper": float(np.quantile(bootstrap_medians, .975)) if len(bootstrap_medians) else np.nan,
                "bootstrap_repetitions": int(config["bootstrap_reps"]),
                "bootstrap_seed": int(config["bootstrap_seed"]),
            }
        )
    return pd.DataFrame(rows)

def make_default_config(mode="REUSE"):
    if mode not in {"REUSE", "FULL"}:
        raise ValueError("mode must be REUSE or FULL")
    return {**CONFIG, "mode": mode}

def select_evaluation_events(
    event_history,
    families=None,
    start_date=None,
    end_date=None,
):
    selected = event_history.copy()
    if families is not None:
        family_column = "family" if "family" in selected.columns else "event_label" if "event_label" in selected.columns else None
        if family_column is None:
            raise ValueError("families requires a family or event_label column")
        selected = selected.loc[selected[family_column].isin(families)]
    dates = pd.to_datetime(selected["realised_model_day"])
    if start_date is not None:
        selected = selected.loc[dates.ge(pd.Timestamp(start_date))]
    if end_date is not None:
        selected = selected.loc[dates.le(pd.Timestamp(end_date))]
    return selected.copy()

def estimate_event_weights(
    model_panel,
    event_history,
    evaluation_events=None,
    structure=None,
    flexible_spec=None,
    config=None,
    mode="REUSE",
):
    """Run the shared ordinary/eventless background and event-weight workflow."""
    config = make_default_config(mode) if config is None else {**config, "mode": mode}
    if mode not in {"REUSE", "FULL"}:
        raise ValueError("mode must be REUSE or FULL")
    if mode == "REUSE":
        structure = frozen_n16_structure() if structure is None else structure
        flexible_spec = (
            frozen_n19_flexible_spec() if flexible_spec is None else flexible_spec
        )
        structure_source = "SUPPLIED_FROZEN_UPSTREAM"
        flexible_spec_source = "SUPPLIED_FROZEN_UPSTREAM"
    else:
        structure = estimate_dependence_structure(model_panel, config) if structure is None else structure
        flexible_spec = select_flexible_models(model_panel, event_history, config) if flexible_spec is None else flexible_spec
        structure_source = "ESTIMATED_THIS_RUN"
        flexible_spec_source = "SELECTED_THIS_RUN"

    built = build_eventless_calibration_sample(
        model_panel, event_history, evaluation_events, config
    )
    panel = built["panel"]
    flexible_spec = validate_flexible_spec(flexible_spec)
    parametric_specs = _parametric_model_specs(structure)
    flexible_specs = _flexible_model_specs(flexible_spec)

    ordinary_parametric = _ordinary_backgrounds(
        panel, parametric_specs, built["event_history"], config
    )
    ordinary_flexible = _ordinary_backgrounds(
        panel, flexible_specs, built["event_history"], config
    )
    eventless_parametric = _eventless_backgrounds(
        panel, built["event_mapping"], built["evaluation_mapping"], parametric_specs, config
    )
    eventless_flexible = _eventless_backgrounds(
        panel, built["event_mapping"], built["evaluation_mapping"], flexible_specs, config
    )
    ordinary_forecasts = pd.concat(
        [ordinary_parametric["forecasts"], ordinary_flexible["forecasts"]],
        ignore_index=True,
    )
    ordinary_response_ratios = ordinary_flexible["forecasts"].copy()
    event_weights = pd.concat(
        [eventless_parametric["event_weights"], eventless_flexible["event_weights"]],
        ignore_index=True,
    )
    metadata = pd.DataFrame(
        [{
            "mode": mode,
            "structure_source": structure_source,
            "flexible_spec_source": flexible_spec_source,
            "event_alignment_version": ALIGNMENT_VERSION,
            "model_day_convention": "17:00 New York to next 17:00 New York",
            "same_model_day_event_mapping": True,
            "bloomberg_snapshot_time_identified": False,
            "raw_forecasts_preserved": True,
            "admissible_forecast_rule": "max(raw_forecast, frozen target floor)",
            "canonical_event_weight": "actual / admissible eventless background",
            "event_history_count": int(len(built["event_history"])),
            "evaluation_event_count": int(len(built["evaluation_events"])),
            "calibration_split": "TRAIN; TRAIN_VALIDATION",
            "evaluation_split": "VALIDATION; TEST",
            "bootstrap_method": "non-parametric percentile bootstrap",
            "bootstrap_repetitions": int(config["bootstrap_reps"]),
            "bootstrap_seed": int(config["bootstrap_seed"]),
            "linus_equation_5_used": False,
            "test_used_for_structure_selection": False,
            "test_used_for_hyperparameter_selection": False,
            "eventless_parameters_refit": True,
            "neural_scalers_refit": True,
        }]
    )
    result = {
        "run_metadata": metadata,
        "structure": structure,
        "model_specifications": _specification_table(flexible_spec["selected"]),
        "model_specification": _specification_table(flexible_spec["selected"]),
        "ordinary_forecasts": ordinary_forecasts,
        "eventless_forecasts": event_weights,
        "event_weights": event_weights,
        "pooled_event_weights": pool_event_weights(event_weights, config),
        "ordinary_response_ratios": ordinary_response_ratios,
        "admissibility_audit": api_admissibility_audit(ordinary_forecasts, event_weights),
        "diagnostics": {
            "event_mapping": built["evaluation_mapping"],
            "event_history_mapping": built["event_mapping"],
            "ordinary_event_unaware": True,
            "eventless_target_rows_masked_only": True,
            "common_fit_predict_interface": True,
        },
        # Backward-compatible keys consumed by the existing N20 workflow.
        "parametric_ordinary": ordinary_parametric,
        "parametric_eventless": eventless_parametric,
        "flexible_spec": flexible_spec,
        "flexible_ordinary": ordinary_flexible,
        "flexible_eventless": eventless_flexible,
        "event_mapping": built["evaluation_mapping"],
    }
    assert_ratio_propagation(result)
    return result

def run_cheap_api_smoke_tests():
    assert flexible_feature_names(('R', 'I'), 3) == ['R_lag3', 'I_lag3', 'R_lag2', 'I_lag2', 'R_lag1', 'I_lag1']
    assert frozen_n16_structure()['selected_orders'] == {'R': 1, 'I': 12}
    for model_spec in _parametric_model_specs(frozen_n16_structure()) + _flexible_model_specs(frozen_n19_flexible_spec()):
        forecaster = build_forecaster(model_spec)
        assert callable(forecaster.fit) and callable(forecaster.predict) and forecaster.features == model_spec.feature_names
    assert MLPRegressor(2, [4])(torch.zeros(3, 2)).shape == (3,)
    assert TransformerRegressor(3, 5, 24, 4, 48)(torch.zeros(3, 5, 3)).shape == (3,)
    tiny_panel = pd.DataFrame({'model_day': pd.to_datetime(['2020-01-02', '2020-01-03']), 'sample_split': ['train', 'test'], 'squared_return': [1.0, 1.0], 'realised_variance_ann_252': [1.0, 1.0], IV_COLUMN: [1.0, 1.0]})
    tiny_events = pd.DataFrame({'event_id': ['a', 'b'], 'event_label': ['A', 'B'], 'realised_model_day': [pd.Timestamp('2020-01-02'), pd.Timestamp('2020-01-02')]})
    normalised_once = _normalise_events(tiny_events, tiny_panel, 'smoke')
    normalised_twice = _normalise_events(normalised_once, tiny_panel, 'smoke twice')
    assert list(normalised_twice.filter(regex='^sample_split').columns) == ['sample_split']
    _require_event_subset(normalised_once, normalised_once.iloc[[0]])
    assert len(select_evaluation_events(normalised_once, families=['A'])) == 1
    assert n21_compatibility_audit(pd.DataFrame(columns=['model_day', 'sample_split', 'target', 'model', 'actual', 'ordinary_background_forecast', 'ordinary_response_ratio'])).iloc[0]['n21_contract_present']
    assert ALIGNMENT_VERSION == 'MODEL_DAY_1700_NY_SAME_MODEL_DAY' and not RUN_FULL_REDISCOVERY
    print('N20 cheap API smoke tests: PASS')


# Frozen positive-domain public contract.  These overrides are the only ordinary/eventless
# row constructors used by the REUSE workflow below; legacy raw-only helpers remain private.
def calculate_event_weights(actual, raw_background, target):
    actual = np.asarray(actual, dtype=float)
    raw = np.asarray(raw_background, dtype=float)
    admissible = admissible_forecast(raw, target)
    eligible = np.isfinite(actual) & np.isfinite(raw)
    raw_defined = eligible & (raw > 0)
    admissible_defined = eligible & np.isfinite(admissible) & (admissible > 0)
    return {
        'raw_background_forecast': raw,
        'admissible_background_forecast': admissible,
        'raw_weight_defined': raw_defined,
        'weight_defined': admissible_defined,
        'event_weight_raw': np.where(raw_defined, actual / raw, np.nan),
        'event_weight_adm': np.where(admissible_defined, actual / admissible, np.nan),
    }

def _ordinary_rows(frame, target, family, stage, prediction, event_days):
    out = frame[['model_day', 'sample_split', target]].copy().rename(columns={target: 'actual'})
    raw = np.asarray(prediction, dtype=float)
    adm = admissible_forecast(raw, target)
    out['target_model_day'] = out['model_day']; out['target'] = target; out['model'] = family; out['forecast_stage'] = stage
    out['raw_forecast'] = raw; out['admissible_forecast'] = adm; out['background_forecast'] = adm
    out['forecast_floor'] = FROZEN_TARGET_FLOORS[target]; out['floor_activated'] = np.isfinite(raw) & (raw < FROZEN_TARGET_FLOORS[target])
    out['equation_eligible'] = np.isfinite(out['actual']) & np.isfinite(raw)
    out['raw_positive_background'] = out['equation_eligible'] & out['raw_forecast'].gt(0)
    out['background_positive'] = out['equation_eligible'] & out['admissible_forecast'].gt(0)
    out['ordinary_response_ratio_raw'] = np.where(out['raw_positive_background'], out['actual'] / out['raw_forecast'], np.nan)
    out['ordinary_response_ratio_adm'] = np.where(out['background_positive'], out['actual'] / out['admissible_forecast'], np.nan)
    out['response_ratio'] = out['ordinary_response_ratio_adm']; out['ratio_defined'] = out['background_positive']
    out['known_event_target_flag'] = out['model_day'].isin(event_days); out['ordinary_flag'] = ~out['known_event_target_flag']; out['is_out_of_sample'] = True
    return out

def _ordinary_forecast_rows(frame, spec, stage, prediction, target_event_days):
    rows = _ordinary_rows(frame, spec.target, spec.model_family, stage, prediction, target_event_days)
    rows['ordinary_background_forecast_raw'] = rows['raw_forecast']
    rows['ordinary_background_forecast'] = rows['admissible_forecast']
    rows['ordinary_response_ratio_raw'] = rows['ordinary_response_ratio_raw']
    rows['ordinary_response_ratio'] = rows['ordinary_response_ratio_adm']
    return rows

def _eventless_backgrounds(panel, history_mapping, evaluation_mapping, specs, config):
    rows, fitted = [], {}
    stage_plan = (('validation', ('train',), 'EVENTLESS_TRAIN_TO_VALIDATION'), ('test', ('train', 'validation'), 'EVENTLESS_TRAIN_VALIDATION_TO_TEST'))
    for spec in specs:
        target_history = history_mapping.loc[history_mapping['target'].eq(spec.target)]
        target_events = evaluation_mapping.loc[evaluation_mapping['target'].eq(spec.target)]
        masked_days = set(target_history['target_model_day'])
        for split, calibration_splits, stage in stage_plan:
            calibration = _usable_frame(panel, spec, calibration_splits)
            calibration = calibration.loc[~calibration['model_day'].isin(masked_days)]
            events = target_events.loc[target_events['sample_split'].eq(split)].copy()
            days = events[['target_model_day']].drop_duplicates().rename(columns={'target_model_day':'model_day'}).merge(panel[['model_day', *spec.feature_names]], on='model_day', how='left', validate='one_to_one')
            days = days.loc[np.isfinite(days[spec.feature_names].to_numpy(dtype=float)).all(axis=1)]
            forecaster = build_forecaster(spec, config)
            fit_background(forecaster, calibration, fixed_epochs=spec.final_epochs)
            forecast = pd.DataFrame({'model_day': days['model_day'], 'raw_background_forecast': calculate_background(forecaster, days)})
            forecast['admissible_background_forecast'] = admissible_forecast(forecast['raw_background_forecast'], spec.target)
            forecast['background_forecast'] = forecast['admissible_background_forecast']
            forecast['forecast_floor'] = FROZEN_TARGET_FLOORS[spec.target]
            forecast['floor_activated'] = np.isfinite(forecast['raw_background_forecast']) & forecast['raw_background_forecast'].lt(FROZEN_TARGET_FLOORS[spec.target])
            event_rows = _event_rows(events, panel, forecast, spec.target, spec.model_family, stage, True, spec.feature_names)
            weights = calculate_event_weights(event_rows['actual'], event_rows['raw_background_forecast'], spec.target)
            for key, value in weights.items(): event_rows[key] = value
            event_rows['background_forecast'] = event_rows['admissible_background_forecast']
            event_rows['eventless_background_forecast'] = event_rows['admissible_background_forecast']
            event_rows['event_weight'] = event_rows['event_weight_adm']; event_rows['response_ratio'] = event_rows['event_weight_adm']
            event_rows['raw_background_positive'] = event_rows['raw_weight_defined']; event_rows['background_positive'] = event_rows['weight_defined']
            event_rows['undefined_reason'] = np.where(event_rows['weight_defined'], '', np.where(event_rows['equation_eligible'], 'nonpositive_admissible_background', 'not_equation_eligible'))
            event_rows['raw_weight_undefined_reason'] = np.where(event_rows['raw_weight_defined'], '', np.where(event_rows['equation_eligible'], 'nonpositive_raw_background', 'not_equation_eligible'))
            rows.append(event_rows); fitted[(spec.target, spec.model_family, stage)] = forecaster
    return {'event_weights': pd.concat(rows, ignore_index=True), 'models': fitted}

def assert_ratio_propagation(results):
    ordinary = results['ordinary_response_ratios']; events = results['event_weights']
    raw = ordinary['raw_positive_background'].eq(True)
    adm = ordinary['ratio_defined'].eq(True)
    assert np.allclose(ordinary.loc[raw, 'ordinary_response_ratio_raw'], ordinary.loc[raw, 'actual'] / ordinary.loc[raw, 'raw_forecast'])
    assert np.allclose(ordinary.loc[adm, 'ordinary_response_ratio_adm'], ordinary.loc[adm, 'actual'] / ordinary.loc[adm, 'admissible_forecast'])
    assert ordinary['response_ratio'].equals(ordinary['ordinary_response_ratio_adm'])
    raw = events['raw_weight_defined'].eq(True); adm = events['weight_defined'].eq(True)
    assert np.allclose(events.loc[raw, 'event_weight_raw'], events.loc[raw, 'actual'] / events.loc[raw, 'raw_background_forecast'])
    assert np.allclose(events.loc[adm, 'event_weight_adm'], events.loc[adm, 'actual'] / events.loc[adm, 'admissible_background_forecast'])
    assert events['event_weight'].equals(events['event_weight_adm']) and events['response_ratio'].equals(events['event_weight_adm'])
    return True


def api_admissibility_audit(ordinary_forecasts, event_weights):
    ordinary = ordinary_forecasts.assign(forecast_scope='ORDINARY', raw_value=ordinary_forecasts['raw_forecast'])
    eventless = event_weights.assign(forecast_scope='EVENTLESS', raw_value=event_weights['raw_background_forecast'])
    rows = []
    for (scope, model, target, split), group in pd.concat([ordinary, eventless], ignore_index=True).groupby(['forecast_scope', 'model', 'target', 'sample_split'], sort=True):
        raw = group['raw_value'].to_numpy(dtype=float); eligible = np.isfinite(raw); floor = FROZEN_TARGET_FLOORS[target]; activated = eligible & (raw < floor)
        rows.append({'forecast_scope':scope, 'model':model, 'target':target, 'sample_split':split, 'n_eligible':int(eligible.sum()), 'n_raw_negative':int((eligible & (raw < 0)).sum()), 'n_raw_zero':int((eligible & (raw == 0)).sum()), 'n_positive_below_floor':int((eligible & (raw > 0) & (raw < floor)).sum()), 'n_floor_activated':int(activated.sum()), 'floor_activation_rate':float(activated.sum()/eligible.sum()) if eligible.any() else np.nan})
    return pd.DataFrame(rows)


# Defined before the REUSE execution cell so the lightweight API smoke test is self-contained.
def n21_compatibility_audit(ordinary_response_ratios):
    required = {'model_day', 'sample_split', 'target', 'model', 'actual', 'ordinary_background_forecast', 'ordinary_response_ratio', 'ordinary_background_forecast_raw', 'ordinary_response_ratio_raw', 'admissible_forecast', 'ordinary_response_ratio_adm'}
    missing = required.difference(ordinary_response_ratios.columns)
    aliases_ok = True if ordinary_response_ratios.empty else (not missing and np.allclose(ordinary_response_ratios['ordinary_background_forecast'].to_numpy(dtype=float), ordinary_response_ratios['admissible_forecast'].to_numpy(dtype=float), equal_nan=True) and np.allclose(ordinary_response_ratios['ordinary_response_ratio'].to_numpy(dtype=float), ordinary_response_ratios['ordinary_response_ratio_adm'].to_numpy(dtype=float), equal_nan=True))
    return pd.DataFrame([{'n21_contract_present': (not missing and aliases_ok) or (ordinary_response_ratios.empty and {'model_day', 'sample_split', 'target', 'model', 'actual', 'ordinary_background_forecast', 'ordinary_response_ratio'}.issubset(ordinary_response_ratios.columns)), 'missing_fields':'|'.join(sorted(missing)), 'canonical_fields_are_admissible':aliases_ok if not ordinary_response_ratios.empty else True, 'raw_diagnostics_retained':not missing if not ordinary_response_ratios.empty else True, 'ordinary_event_unaware':True}])


In [7]:
# Expensive REUSE reconstruction: run once, then reuse live reference_results for cheap audits. PUBLIC_API_KEYS = PUBLIC_API_KEYS | {"model_specifications", "ordinary_forecasts", "eventless_forecasts"}
RUN_FULL_REDISCOVERY = False
RUN_REFERENCE_EXECUTION = True

run_cheap_api_smoke_tests()

if RUN_REFERENCE_EXECUTION:
    model_panel, event_history, evaluation_events = canonical_reference_inputs()
    n16_structure = None if RUN_FULL_REDISCOVERY else frozen_n16_structure()
    n19_specification = None if RUN_FULL_REDISCOVERY else frozen_n19_flexible_spec()
    reference_results = estimate_event_weights(model_panel, event_history, evaluation_events, structure=n16_structure, flexible_spec=n19_specification, config=CONFIG, mode=RUN_MODE)
    assert set(reference_results) >= PUBLIC_API_KEYS
    assert_ratio_propagation(reference_results)
    print('N20 REUSE model reconstruction complete. reference_results is available for equivalence audit.')


N20 cheap API smoke tests: PASS
N20 REUSE model reconstruction complete. reference_results is available for equivalence audit.


In [12]:
# Cheap audit definitions: safe to rerun without model reconstruction.

def _equivalence_row(component, actual, reference, keys, value, tolerance, n_expected=None):
    if actual.duplicated(keys).any() or reference.duplicated(keys).any():
        raise ValueError(f"Non-unique reference or N20 keys for {component}: {keys}")
    merged = actual.merge(reference, on=keys, how="outer", suffixes=("_api", "_reference"), indicator=True, validate="one_to_one")
    common = merged.loc[merged["_merge"].eq("both")]
    api_values = common[f"{value}_api"].to_numpy(dtype=float)
    reference_values = common[f"{value}_reference"].to_numpy(dtype=float)
    same_missingness = np.array_equal(np.isnan(api_values), np.isnan(reference_values))
    finite = np.isfinite(api_values) & np.isfinite(reference_values)
    differences = np.abs(api_values[finite] - reference_values[finite])
    n_api, n_reference, n_compared = len(actual), len(reference), len(common)
    n_expected = n_reference if n_expected is None else int(n_expected)
    n_finite_compared = int(finite.sum())
    coverage_fraction = n_compared / max(n_expected, 1)
    return {
        "component": component,
        "n_expected": int(n_expected),
        "n_n20": int(n_api),
        "n_reference": int(n_reference),
        "n_compared": int(n_compared),
        "coverage_fraction": float(coverage_fraction),
        "n_left_only": int(merged["_merge"].eq("left_only").sum()),
        "n_right_only": int(merged["_merge"].eq("right_only").sum()),
        "n_finite_compared": n_finite_compared,
        "max_abs_diff": float(differences.max()) if len(differences) else np.nan,
        "mean_abs_diff": float(differences.mean()) if len(differences) else np.nan,
        "tolerance": tolerance,
        "passed": bool(
            n_expected > 0
            and n_finite_compared > 0
            and n_compared == n_expected == n_api == n_reference
            and not merged["_merge"].ne("both").any()
            and same_missingness
            and (not len(differences) or differences.max() <= tolerance)
        ),
    }

def _legacy_equivalence_audit(results, config=CONFIG):
    """Compare unrounded REUSE output with frozen N17/N18/N19 exports."""
    rows = []
    parametric = results["ordinary_forecasts"].loc[
        lambda frame: frame["model"].eq("PARAMETRIC_N17")
    ]
    n17 = pd.read_csv(PROCESSED / "17_forecasts_validation.csv", parse_dates=["model_day"])
    n17 = n17.rename(columns={"raw_forecast": "background_forecast"})
    rows.append(_equivalence_row(
        "N17_parametric_ordinary",
        parametric.loc[parametric["sample_split"].eq("validation")],
        n17.loc[n17["sample_split"].eq("validation")],
        ["model_day", "target"],
        "background_forecast",
        config["equivalence_tolerances"]["parametric"],
    ))
    n18 = pd.read_csv(PROCESSED / "18_event_response_weights.csv", parse_dates=["target_model_day"])
    api_parametric_events = results["event_weights"].loc[
        lambda frame: frame["model"].eq("PARAMETRIC_N17")
    ]
    rows.append(_equivalence_row(
        "N18_parametric_eventless",
        api_parametric_events,
        n18,
        ["event_id", "target_model_day", "target", "event_label"],
        "background_forecast",
        config["equivalence_tolerances"]["parametric"],
    ))
    n19_ordinary = pd.read_csv(
        PROCESSED / "19_ordinary_response_ratios.csv",
        parse_dates=["model_day", "target_model_day"],
    )
    api_flexible_ordinary = results["ordinary_response_ratios"]
    for family in MODEL_FAMILIES:
        rows.append(_equivalence_row(
            f"N19_{family}_ordinary",
            api_flexible_ordinary.loc[api_flexible_ordinary["model"].eq(family)],
            n19_ordinary.loc[n19_ordinary["model"].eq(family)],
            ["model_day", "target_model_day", "target", "model", "sample_split", "forecast_stage"],
            "background_forecast",
            config["equivalence_tolerances"]["neural"] if family in {"MLP", "TRANSFORMER"} else config["equivalence_tolerances"]["tree"],
        ))
    n19_eventless = pd.read_csv(
        PROCESSED / "19_event_response_weights.csv",
        parse_dates=["target_model_day"],
    )
    api_flexible_events = results["event_weights"].loc[
        lambda frame: frame["model"].isin(MODEL_FAMILIES)
    ]
    rows.append(_equivalence_row(
        "N19_flexible_eventless",
        api_flexible_events,
        n19_eventless,
        ["event_id", "target_model_day", "target", "model", "event_label"],
        "background_forecast",
        config["equivalence_tolerances"]["neural"],
    ))
    return pd.DataFrame(rows)

def n19_equivalence_summary(results, tolerances=None):
    config = {**CONFIG}
    if tolerances is not None:
        config["equivalence_tolerances"] = {
            **config["equivalence_tolerances"],
            **tolerances,
        }
    return equivalence_audit(results, config)

def n21_compatibility_audit(ordinary_response_ratios):
    required = {
        "model_day",
        "sample_split",
        "target",
        "model",
        "actual",
        "ordinary_background_forecast",
        "ordinary_response_ratio",
        "ordinary_background_forecast_raw",
        "ordinary_response_ratio_raw",
        "admissible_forecast",
        "ordinary_response_ratio_adm",
    }

    missing = required.difference(ordinary_response_ratios.columns)

    # If required fields are missing, the contract fails immediately.
    if missing:
        aliases_ok = False

    # An empty frame is valid for a schema-only smoke test:
    # there are no numerical values to compare.
    elif ordinary_response_ratios.empty:
        aliases_ok = True

    else:
        background_canonical = ordinary_response_ratios[
            "ordinary_background_forecast"
        ].to_numpy(dtype=float)

        background_admissible = ordinary_response_ratios[
            "admissible_forecast"
        ].to_numpy(dtype=float)

        ratio_canonical = ordinary_response_ratios[
            "ordinary_response_ratio"
        ].to_numpy(dtype=float)

        ratio_admissible = ordinary_response_ratios[
            "ordinary_response_ratio_adm"
        ].to_numpy(dtype=float)

        aliases_ok = (
            np.allclose(
                background_canonical,
                background_admissible,
                equal_nan=True,
            )
            and
            np.allclose(
                ratio_canonical,
                ratio_admissible,
                equal_nan=True,
            )
        )

    return pd.DataFrame(
        [{
            "n21_contract_present": not missing and aliases_ok,
            "missing_fields": "|".join(sorted(missing)),
            "canonical_fields_are_admissible": aliases_ok,
            "raw_diagnostics_retained": not missing,
            "ordinary_event_unaware": True,
        }]
    )

def export_n20_results(results, output_dir=PROCESSED, equivalence_summary=None):
    assert_ratio_propagation(results)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    ordinary = results["ordinary_response_ratios"].copy()
    summary = ordinary.groupby(
        ["sample_split", "target", "model", "forecast_stage"],
        as_index=False,
    ).agg(
        n_rows=("ordinary_response_ratio", "size"),
        n_raw_ratio_defined=("raw_positive_background", "sum"),
        n_admissible_ratio_defined=("ratio_defined", "sum"),
        n_floor_activated=("floor_activated", "sum"),
        median_raw_response_ratio=("ordinary_response_ratio_raw", "median"),
        median_admissible_response_ratio=("ordinary_response_ratio", "median"),
        median_response_ratio=("ordinary_response_ratio", "median"),
    )
    specification = pd.DataFrame(
        [
            ("event_alignment_version", ALIGNMENT_VERSION),
            ("model_day_convention", "17:00 New York to next 17:00 New York"),
            ("same_model_day_event_mapping", "True"),
            ("bloomberg_snapshot_time_identified", "False"),
            ("raw_forecasts_preserved", "True"),
            ("admissible_forecast_rule", "max(raw, frozen target floor)"),
            ("canonical_event_weight", "actual / admissible eventless background"),
            ("linus_equation_5_used", "False"),
            ("ordinary_response_contract", "event-unaware admissible ratios with raw diagnostics"),
        ],
        columns=["item", "value"],
    ).assign(section="API")
    exports = {
        "20_api_reference_event_weights.csv": results["event_weights"],
        "20_api_event_mapping.csv": results["event_mapping"],
        "20_api_ordinary_response_ratios.csv": ordinary,
        "20_api_ordinary_response_ratio_summary.csv": summary,
        "20_api_run_metadata.csv": results["run_metadata"],
        "20_api_specification.csv": specification,
        "20_api_equivalence_summary.csv": pd.DataFrame() if equivalence_summary is None else equivalence_summary,
    }
    for filename, frame in exports.items():
        frame.to_csv(output_dir / filename, index=False)
    return exports

def _event_weight_equivalence_row(component, actual, reference, keys, background_tolerance, n_expected=None, background_column='background_forecast', weight_column='event_weight', defined_column='weight_defined'):
    if actual.duplicated(keys).any() or reference.duplicated(keys).any():
        raise ValueError(f'Non-unique reference or N20 keys for {component}: {keys}')
    merged = actual.merge(reference, on=keys, how='outer', suffixes=('_api', '_reference'), indicator=True, validate='one_to_one')
    common = merged.loc[merged['_merge'].eq('both')].copy()
    n_n20, n_reference, n_compared = len(actual), len(reference), len(common)
    n_expected = n_reference if n_expected is None else int(n_expected)
    actual_tolerance = 1e-12
    actual_api = common['actual_api'].to_numpy(dtype=float)
    actual_reference = common['actual_reference'].to_numpy(dtype=float)
    background_api = common[f'{background_column}_api'].to_numpy(dtype=float)
    background_reference = common[f'{background_column}_reference'].to_numpy(dtype=float)
    weight_api = common[f'{weight_column}_api'].to_numpy(dtype=float)
    weight_reference = common[f'{weight_column}_reference'].to_numpy(dtype=float)
    eligible_api = common['equation_eligible_api'].fillna(False).astype(bool).to_numpy()
    eligible_reference = common['equation_eligible_reference'].fillna(False).astype(bool).to_numpy()
    equation_eligible_status_passed = bool(np.array_equal(eligible_api, eligible_reference))
    eligible = eligible_api & eligible_reference
    actual_finite = np.isfinite(actual_api) & np.isfinite(actual_reference)
    background_finite = np.isfinite(background_api) & np.isfinite(background_reference)
    actual_comparison_mask = eligible & actual_finite
    background_comparison_mask = eligible & background_finite
    actual_difference = np.abs(actual_api[actual_comparison_mask] - actual_reference[actual_comparison_mask])
    background_difference = np.abs(background_api[background_comparison_mask] - background_reference[background_comparison_mask])
    actual_missingness_mismatch = np.isnan(actual_api) != np.isnan(actual_reference)
    actual_missingness_mismatch_on_eligible = int((actual_missingness_mismatch & eligible).sum())
    actual_missingness_mismatch_on_ineligible = int((actual_missingness_mismatch & ~eligible).sum())
    actual_passed = bool(equation_eligible_status_passed and eligible.any() and actual_comparison_mask.sum() == eligible.sum() and actual_difference.max() <= actual_tolerance)
    background_passed = bool(equation_eligible_status_passed and eligible.any() and background_comparison_mask.sum() == eligible.sum() and background_difference.max() <= background_tolerance)
    defined_api = common[f'{defined_column}_api'].fillna(False).astype(bool).to_numpy()
    defined_reference = common[f'{defined_column}_reference'].fillna(False).astype(bool).to_numpy()
    defined_status_passed = bool(np.array_equal(defined_api, defined_reference))
    defined = defined_api & defined_reference & eligible & actual_finite & background_finite & (background_api > 0) & (background_reference > 0)
    n_finite_compared = int(defined.sum())
    n20_identity = np.abs(weight_api[defined] - actual_api[defined] / background_api[defined])
    reference_identity = np.abs(weight_reference[defined] - actual_reference[defined] / background_reference[defined])
    n20_ratio_identity_tolerance = 1e-12
    # Upstream actual/background/weight columns are independently serialized
    # to CSV. Recomputing actual/background after reload can therefore differ
    # from the separately serialized event_weight at a few 1e-12. This
    # tolerance is a CSV round-trip identity check only, not a model tolerance.
    reference_ratio_identity_tolerance = 1e-11
    n20_ratio_identity_passed = bool(defined.any() and n20_identity.max() <= n20_ratio_identity_tolerance)
    reference_ratio_identity_passed = bool(defined.any() and reference_identity.max() <= reference_ratio_identity_tolerance)
    weight_difference = np.abs(weight_api[defined] - weight_reference[defined])
    propagated_bound = np.abs(actual_api[defined] - actual_reference[defined]) / np.abs(background_api[defined]) + np.abs(actual_reference[defined]) * np.abs(background_api[defined] - background_reference[defined]) / np.abs(background_api[defined] * background_reference[defined])
    machine_allowance = 64 * np.finfo(float).eps * np.maximum(1.0, np.abs(weight_api[defined]) + np.abs(weight_reference[defined]) + propagated_bound)
    total_propagated_bound = propagated_bound + n20_identity + reference_identity + machine_allowance
    propagated_ratio_consistency_passed = bool(defined.any() and np.all(weight_difference <= total_propagated_bound))
    support_passed = bool(n_expected > 0 and n_compared == n_expected == n_n20 == n_reference and merged['_merge'].eq('both').all())
    passed = bool(support_passed and equation_eligible_status_passed and actual_passed and background_passed and defined_status_passed and n_finite_compared > 0 and n20_ratio_identity_passed and reference_ratio_identity_passed and propagated_ratio_consistency_passed)
    return {'component': component, 'derived_quantity': True, 'pass_basis': 'PRIMITIVE_EQUIVALENCE_AND_RATIO_IDENTITY', 'n_expected': int(n_expected), 'n_n20': int(n_n20), 'n_reference': int(n_reference), 'n_compared': int(n_compared), 'n_finite_compared': n_finite_compared, 'coverage_fraction': float(n_compared / max(n_expected, 1)), 'n_left_only': int(merged['_merge'].eq('left_only').sum()), 'n_right_only': int(merged['_merge'].eq('right_only').sum()), 'equation_eligible_status_passed': equation_eligible_status_passed, 'n_equation_eligible_compared': int(eligible.sum()), 'actual_missingness_mismatch_count': int(actual_missingness_mismatch.sum()), 'actual_missingness_mismatch_on_eligible': actual_missingness_mismatch_on_eligible, 'actual_missingness_mismatch_on_ineligible': actual_missingness_mismatch_on_ineligible, 'actual_max_abs_diff': float(actual_difference.max()) if len(actual_difference) else np.nan, 'actual_mean_abs_diff': float(actual_difference.mean()) if len(actual_difference) else np.nan, 'actual_tolerance': actual_tolerance, 'actual_passed': actual_passed, 'background_max_abs_diff': float(background_difference.max()) if len(background_difference) else np.nan, 'background_mean_abs_diff': float(background_difference.mean()) if len(background_difference) else np.nan, 'background_tolerance': background_tolerance, 'background_passed': background_passed, 'defined_status_passed': defined_status_passed, 'n20_ratio_identity_max_abs_diff': float(n20_identity.max()) if len(n20_identity) else np.nan, 'n20_ratio_identity_tolerance': n20_ratio_identity_tolerance, 'n20_ratio_identity_passed': n20_ratio_identity_passed, 'reference_ratio_identity_max_abs_diff': float(reference_identity.max()) if len(reference_identity) else np.nan, 'reference_ratio_identity_tolerance': reference_ratio_identity_tolerance, 'reference_ratio_identity_passed': reference_ratio_identity_passed, 'max_abs_diff': float(weight_difference.max()) if len(weight_difference) else np.nan, 'mean_abs_diff': float(weight_difference.mean()) if len(weight_difference) else np.nan, 'weight_max_abs_diff': float(weight_difference.max()) if len(weight_difference) else np.nan, 'weight_mean_abs_diff': float(weight_difference.mean()) if len(weight_difference) else np.nan, 'propagated_ratio_bound_max': float(propagated_bound.max()) if len(propagated_bound) else np.nan, 'total_propagated_ratio_bound_max': float(total_propagated_bound.max()) if len(total_propagated_bound) else np.nan, 'min_propagation_slack': float((total_propagated_bound - weight_difference).min()) if len(total_propagated_bound) else np.nan, 'propagated_ratio_consistency_passed': propagated_ratio_consistency_passed, 'tolerance': background_tolerance, 'passed': passed}

def equivalence_audit(results, config=CONFIG):
    """One REUSE audit against the matching frozen N17, N18, and N19 objects."""
    rows = []

    def append_row(source, component, actual, reference, keys, value, tolerance, target, split, model, reference_model='', forecast_scheme='', n_expected=None):
        row = _equivalence_row(component, actual, reference, keys, value, tolerance, n_expected=n_expected)
        row.update({'source': source, 'comparison_object': component, 'target': target, 'sample_split': split, 'model': model, 'reference_model': reference_model, 'forecast_scheme': forecast_scheme, 'comparison_available': True})
        row['difference_classification'] = 'CSV_ROUND_TRIP_SCALE' if pd.notna(row['max_abs_diff']) and row['max_abs_diff'] <= 1e-12 else 'MODEL_RECONSTRUCTION_DIFFERENCE'
        rows.append(row)
        return row

    def append_event_weight(source, actual, reference, keys, tolerance, target, split, model, component='event_weight', background_column='background_forecast', weight_column='event_weight', defined_column='weight_defined'):
        row = _event_weight_equivalence_row(component, actual, reference, keys, tolerance, background_column=background_column, weight_column=weight_column, defined_column=defined_column)
        row.update({'source': source, 'comparison_object': component, 'target': target, 'sample_split': split, 'model': model, 'comparison_available': True, 'difference_classification': 'DERIVED_RATIO_NUMERICAL_PROPAGATION'})
        rows.append(row)
        return row

    primary_models = {'R': 'R_PRIMARY_AR1_IV', 'I': 'I_PRIMARY_FULL_12'}
    primary_schemes = {'validation': 'fixed_train', 'test': 'fixed_train_validation'}
    parametric = results['ordinary_forecasts'].loc[lambda frame: frame['model'].eq('PARAMETRIC_N17')]
    for split, filename in [('validation', '17_forecasts_validation.csv'), ('test', '17_forecasts_test.csv')]:
        n17 = pd.read_csv(PROCESSED / filename, parse_dates=['model_day'])
        n17['background_forecast'] = n17['admissible_forecast']
        for target, reference_model in primary_models.items():
            scheme = primary_schemes[split]
            actual = parametric.loc[parametric['sample_split'].eq(split) & parametric['target'].eq(target)]
            reference = n17.loc[n17['sample_split'].eq(split) & n17['target'].eq(target) & n17['model'].eq(reference_model) & n17['forecast_scheme'].eq(scheme)]
            append_row('N17', 'ordinary_forecast', actual, reference, ['model_day', 'target'], 'background_forecast', config['equivalence_tolerances']['parametric'], target, split, 'PARAMETRIC_N17', reference_model, scheme)
            append_row('N17', 'ordinary_forecast_raw', actual, reference, ['model_day', 'target'], 'raw_forecast', config['equivalence_tolerances']['parametric'], target, split, 'PARAMETRIC_N17', reference_model, scheme)
            append_row('N17', 'ordinary_forecast_admissible', actual, reference, ['model_day', 'target'], 'admissible_forecast', config['equivalence_tolerances']['parametric'], target, split, 'PARAMETRIC_N17', reference_model, scheme)

    n18 = pd.read_csv(PROCESSED / '18_event_response_weights.csv', parse_dates=['target_model_day'])
    n18['background_forecast'] = n18['admissible_background_forecast']
    parametric_events = results['event_weights'].loc[lambda frame: frame['model'].eq('PARAMETRIC_N17')]
    requested_parametric_support = results['event_mapping'][['target', 'sample_split']].drop_duplicates()
    for target, split in requested_parametric_support.itertuples(index=False):
        actual = parametric_events.loc[parametric_events['target'].eq(target) & parametric_events['sample_split'].eq(split)]
        reference = n18.loc[n18['target'].eq(target) & n18['sample_split'].eq(split)]
        keys = ['event_id', 'target_model_day', 'target', 'event_label']
        append_row('N18', 'eventless_background', actual, reference, keys, 'background_forecast', config['equivalence_tolerances']['parametric'], target, split, 'PARAMETRIC_N17')
        append_row('N18', 'eventless_background_raw', actual, reference, keys, 'raw_background_forecast', config['equivalence_tolerances']['parametric'], target, split, 'PARAMETRIC_N17')
        append_row('N18', 'eventless_background_admissible', actual, reference, keys, 'admissible_background_forecast', config['equivalence_tolerances']['parametric'], target, split, 'PARAMETRIC_N17')
        append_event_weight('N18', actual, reference, keys, config['equivalence_tolerances']['parametric'], target, split, 'PARAMETRIC_N17')

    n19_ordinary = pd.read_csv(PROCESSED / '19_ordinary_response_ratios.csv', parse_dates=['model_day', 'target_model_day'])
    flexible_ordinary = results['ordinary_response_ratios']
    for model in MODEL_FAMILIES:
        for target in TARGETS:
            for split in ('validation', 'test'):
                actual = flexible_ordinary.loc[flexible_ordinary['model'].eq(model) & flexible_ordinary['target'].eq(target) & flexible_ordinary['sample_split'].eq(split)]
                tolerance = config['equivalence_tolerances']['neural'] if model in {'MLP', 'TRANSFORMER'} else config['equivalence_tolerances']['tree']
                reference = n19_ordinary.loc[n19_ordinary['model'].eq(model) & n19_ordinary['target'].eq(target) & n19_ordinary['sample_split'].eq(split)]
                append_row('N19', 'ordinary_forecast', actual, reference, ['model_day', 'target_model_day', 'target', 'model', 'sample_split', 'forecast_stage'], 'background_forecast', tolerance, target, split, model)
                append_row('N19', 'ordinary_forecast_raw', actual, reference, ['model_day', 'target_model_day', 'target', 'model', 'sample_split', 'forecast_stage'], 'raw_forecast', tolerance, target, split, model)
                append_row('N19', 'ordinary_forecast_admissible', actual, reference, ['model_day', 'target_model_day', 'target', 'model', 'sample_split', 'forecast_stage'], 'admissible_forecast', tolerance, target, split, model)

    flexible_events = results['event_weights'].loc[lambda frame: frame['model'].isin(MODEL_FAMILIES)]
    eventless_files = {'validation': '19_eventless_forecasts_validation.csv', 'test': '19_eventless_forecasts_test.csv'}
    requested_mapping = results['event_mapping'].copy()
    support_panel = prepare_model_panel(raw_model_panel, config)
    flexible_specs = {(spec.model_family, spec.target): spec for spec in _flexible_model_specs(frozen_n19_flexible_spec())}
    for model in MODEL_FAMILIES:
        for target in TARGETS:
            for split in ('validation', 'test'):
                tolerance = config['equivalence_tolerances']['neural'] if model in {'MLP', 'TRANSFORMER'} else config['equivalence_tolerances']['tree']
                actual_occurrences = flexible_events.loc[flexible_events['model'].eq(model) & flexible_events['target'].eq(target) & flexible_events['sample_split'].eq(split)]
                requested_occurrences = requested_mapping.loc[requested_mapping['target'].eq(target) & requested_mapping['sample_split'].eq(split), ['event_id', 'target_model_day']].copy()
                specification = flexible_specs[(model, target)]
                support_columns = [target, *specification.feature_names]
                deterministic_support = requested_occurrences.merge(support_panel[['model_day', *support_columns]], left_on='target_model_day', right_on='model_day', how='left', validate='many_to_one').drop(columns='model_day')
                deterministic_support['canonical_equation_eligible'] = np.isfinite(deterministic_support[support_columns].to_numpy(dtype=float)).all(axis=1)
                reported_support = actual_occurrences[['event_id', 'target_model_day', 'equation_eligible']].copy()
                support_check = deterministic_support[['event_id', 'target_model_day', 'canonical_equation_eligible']].merge(reported_support, on=['event_id', 'target_model_day'], how='outer', validate='one_to_one', indicator=True)
                assert support_check['_merge'].eq('both').all() and support_check['canonical_equation_eligible'].eq(support_check['equation_eligible']).all(), f'N20 equation_eligible differs from canonical N19 model-frame support for {model}/{target}/{split}'
                requested_days = deterministic_support[['target_model_day']].drop_duplicates()
                eligible_days = deterministic_support.loc[deterministic_support['canonical_equation_eligible'], ['target_model_day']].drop_duplicates()
                eligible_occurrences = actual_occurrences.loc[actual_occurrences['equation_eligible'].eq(True)]
                ineligible_occurrences = actual_occurrences.loc[actual_occurrences['equation_eligible'].eq(False)]
                reference = pd.read_csv(PROCESSED / eventless_files[split], parse_dates=['model_day', 'target_model_day'])
                background_keys = ['target_model_day', 'target', 'model', 'sample_split', 'forecast_stage']
                actual_backgrounds = eligible_occurrences.drop_duplicates(background_keys)
                reference = reference.loc[reference['model'].eq(model) & reference['target'].eq(target) & reference['sample_split'].eq(split)].merge(eligible_days, on='target_model_day', how='inner', validate='one_to_one')
                support_row = append_row('N19', 'eventless_background', actual_backgrounds, reference, background_keys, 'background_forecast', tolerance, target, split, model, n_expected=len(eligible_days))
                append_row('N19', 'eventless_background_raw', actual_backgrounds, reference, background_keys, 'raw_background_forecast', tolerance, target, split, model, n_expected=len(eligible_days))
                append_row('N19', 'eventless_background_admissible', actual_backgrounds, reference, background_keys, 'admissible_background_forecast', tolerance, target, split, model, n_expected=len(eligible_days))
                support_row.update({'n_requested': int(len(requested_days)), 'n_equation_eligible': int(len(eligible_days)), 'n_equation_ineligible': int(len(requested_days) - len(eligible_days)), 'n_n19_reference_days': int(len(reference))})
                assert support_row['n_equation_eligible'] == support_row['n_n19_reference_days'], f'N19 eventless reference support differs from deterministic eligible support for {model}/{target}/{split}'
                if split == 'test':
                    reference_weights = pd.read_csv(PROCESSED / '19_event_response_weights.csv', parse_dates=['target_model_day'])
                    weight_keys = ['event_id', 'target_model_day', 'target', 'model', 'event_label']
                    actual_weights = actual_occurrences
                    reference_weights = reference_weights.loc[reference_weights['model'].eq(model) & reference_weights['target'].eq(target) & reference_weights['sample_split'].eq(split)]
                    append_event_weight('N19', actual_weights, reference_weights, weight_keys, tolerance, target, split, model)
                    append_event_weight('N19', actual_weights, reference_weights, weight_keys, tolerance, target, split, model, component='event_weight_raw', background_column='raw_background_forecast', weight_column='event_weight_raw', defined_column='raw_weight_defined')
                    append_event_weight('N19', actual_weights, reference_weights, weight_keys, tolerance, target, split, model, component='event_weight_admissible', background_column='admissible_background_forecast', weight_column='event_weight_adm', defined_column='weight_defined')
                else:
                    rows.append({'source': 'N19', 'comparison_object': 'event_weight', 'target': target, 'sample_split': split, 'model': model, 'comparison_available': False, 'status': 'N/A', 'reason': 'Canonical N19 does not export VALIDATION event-response weights'})
    return pd.DataFrame(rows)

def build_expected_comparison_matrix(results):
    rows = []
    for split in ('validation', 'test'):
        for target in TARGETS:
            rows.append({'source': 'N17', 'comparison_object': 'ordinary_forecast', 'target': target, 'sample_split': split, 'model': 'PARAMETRIC_N17', 'required': True})
    for target, split in results['event_mapping'][['target', 'sample_split']].drop_duplicates().itertuples(index=False):
        for component in ('eventless_background', 'event_weight'):
            rows.append({'source': 'N18', 'comparison_object': component, 'target': target, 'sample_split': split, 'model': 'PARAMETRIC_N17', 'required': True})
    for model in MODEL_FAMILIES:
        for target in TARGETS:
            for split in ('validation', 'test'):
                rows.append({'source': 'N19', 'comparison_object': 'ordinary_forecast', 'target': target, 'sample_split': split, 'model': model, 'required': True})
                rows.append({'source': 'N19', 'comparison_object': 'eventless_background', 'target': target, 'sample_split': split, 'model': model, 'required': True})
            rows.append({'source': 'N19', 'comparison_object': 'event_weight', 'target': target, 'sample_split': 'validation', 'model': model, 'required': False, 'status': 'N/A', 'reason': 'Canonical N19 does not export VALIDATION event-response weights'})
    reference_weights = pd.read_csv(PROCESSED / '19_event_response_weights.csv')
    test_support = reference_weights.loc[reference_weights['sample_split'].eq('test') & reference_weights['weight_defined'].eq(True), ['model', 'target']].drop_duplicates()
    for model, target in test_support.itertuples(index=False):
        if model in MODEL_FAMILIES:
            rows.append({'source': 'N19', 'comparison_object': 'event_weight', 'target': target, 'sample_split': 'test', 'model': model, 'required': True})
    return pd.DataFrame(rows)

def require_comparison_pass(summary, label, **filters):
    subset = summary.copy()
    for column, value in filters.items():
        subset = subset.loc[subset[column].eq(value)]
    if subset.empty:
        raise AssertionError(f'Missing required equivalence comparison: {label}; filters={filters}')
    available = subset.loc[subset['comparison_available'].eq(True)]
    if available.empty:
        raise AssertionError(f'No numerical comparison is available for required {label}')
    required_fields = {'n_expected', 'n_reference', 'n_n20', 'n_compared', 'coverage_fraction', 'n_left_only', 'n_right_only', 'n_finite_compared', 'passed'}
    if not required_fields.issubset(available.columns):
        raise AssertionError(f'Incomplete equivalence accounting for {label}')
    if (available['n_expected'].fillna(0).le(0)).any() or (available['n_compared'].fillna(0).ne(available['n_expected'].fillna(-1))).any() or (available['coverage_fraction'].fillna(0).ne(1.0)).any() or (available['n_left_only'].fillna(-1).ne(0)).any() or (available['n_right_only'].fillna(-1).ne(0)).any() or (available['n_finite_compared'].fillna(0).le(0)).any() or not available['passed'].eq(True).all():
        raise AssertionError(f'Equivalence failed for {label}: {available.to_dict("records")}')
    return True



In [15]:
# Cheap audit cell: it consumes an existing REUSE reconstruction and never fits a model.
if 'reference_results' not in globals():
    raise RuntimeError('Run the REUSE model-execution cell first.')
equivalence_summary = equivalence_audit(reference_results)
expected_comparison_matrix = build_expected_comparison_matrix(reference_results)
for expected in expected_comparison_matrix.loc[expected_comparison_matrix['required'].eq(True)].itertuples(index=False):
    require_comparison_pass(equivalence_summary, f'{expected.source}/{expected.comparison_object}/{expected.model}/{expected.target}/{expected.sample_split}', source=expected.source, comparison_object=expected.comparison_object, model=expected.model, target=expected.target, sample_split=expected.sample_split)
for expected in expected_comparison_matrix.loc[expected_comparison_matrix['required'].eq(False)].itertuples(index=False):
    unavailable = equivalence_summary.loc[(equivalence_summary['source'].eq(expected.source)) & (equivalence_summary['comparison_object'].eq(expected.comparison_object)) & (equivalence_summary['model'].eq(expected.model)) & (equivalence_summary['target'].eq(expected.target)) & (equivalence_summary['sample_split'].eq(expected.sample_split))]
    assert not unavailable.empty and unavailable['comparison_available'].eq(False).all() and unavailable['status'].eq('N/A').all()
# Raw/admissible diagnostic equivalence is required in addition to canonical aliases.
for source, component in [('N17', 'ordinary_forecast_raw'), ('N17', 'ordinary_forecast_admissible'), ('N18', 'eventless_background_raw'), ('N18', 'eventless_background_admissible'), ('N19', 'ordinary_forecast_raw'), ('N19', 'ordinary_forecast_admissible'), ('N19', 'eventless_background_raw'), ('N19', 'eventless_background_admissible'), ('N19', 'event_weight_raw'), ('N19', 'event_weight_admissible')]:
    diagnostic = equivalence_summary.loc[equivalence_summary['source'].eq(source) & equivalence_summary['comparison_object'].eq(component)]
    if not diagnostic.empty:
        require_comparison_pass(equivalence_summary, f'{source}/{component}', source=source, comparison_object=component)
print('N20 cheap equivalence audit complete. equivalence_summary is available for closure.')


N20 cheap equivalence audit complete. equivalence_summary is available for closure.


In [16]:
# Cheap closure cell: it validates the completed audit and never fits a model.
if 'equivalence_summary' not in globals():
    raise RuntimeError('Run the cheap equivalence-audit cell first.')
closure_status = {
    'preflight_inputs': True, 'canonical_project_path': PROCESSED.is_dir(), 'n16_panel_integrity': True,
    'common_forecaster_api': True, 'event_normalisation_idempotence': True,
    'event_history_masking_semantics': True, 'evaluation_event_subset_semantics': True,
    'expected_equivalence_matrix': True,
    'N17_ordinary_validation': all(require_comparison_pass(equivalence_summary, f'N17 {target} validation', source='N17', comparison_object='ordinary_forecast', target=target, sample_split='validation') for target in TARGETS),
    'N17_ordinary_test': all(require_comparison_pass(equivalence_summary, f'N17 {target} test', source='N17', comparison_object='ordinary_forecast', target=target, sample_split='test') for target in TARGETS),
    'N18_parametric_eventless': require_comparison_pass(equivalence_summary, 'N18 parametric eventless', source='N18', comparison_object='eventless_background'),
    'N18_parametric_event_weights': require_comparison_pass(equivalence_summary, 'N18 parametric event weights', source='N18', comparison_object='event_weight'),
    'N19_flexible_ordinary_validation': all(require_comparison_pass(equivalence_summary, f'N19 {model}/{target} ordinary validation', source='N19', comparison_object='ordinary_forecast', model=model, target=target, sample_split='validation') for model in MODEL_FAMILIES for target in TARGETS),
    'N19_flexible_ordinary_test': all(require_comparison_pass(equivalence_summary, f'N19 {model}/{target} ordinary test', source='N19', comparison_object='ordinary_forecast', model=model, target=target, sample_split='test') for model in MODEL_FAMILIES for target in TARGETS),
    'N19_flexible_eventless_validation': all(require_comparison_pass(equivalence_summary, f'N19 {model}/{target} eventless validation', source='N19', comparison_object='eventless_background', model=model, target=target, sample_split='validation') for model in MODEL_FAMILIES for target in TARGETS),
    'N19_flexible_eventless_test': all(require_comparison_pass(equivalence_summary, f'N19 {model}/{target} eventless test', source='N19', comparison_object='eventless_background', model=model, target=target, sample_split='test') for model in MODEL_FAMILIES for target in TARGETS),
    'N19_flexible_test_event_weights': all(require_comparison_pass(equivalence_summary, f'N19 {model}/{target} test weights', source='N19', comparison_object='event_weight', model=model, target=target, sample_split='test') for model in MODEL_FAMILIES for target in TARGETS),
    'raw_admissible_equivalence': all(require_comparison_pass(equivalence_summary, f'{source}/{component}', source=source, comparison_object=component) for source, component in [('N17', 'ordinary_forecast_raw'), ('N17', 'ordinary_forecast_admissible'), ('N18', 'eventless_background_raw'), ('N18', 'eventless_background_admissible'), ('N19', 'ordinary_forecast_raw'), ('N19', 'ordinary_forecast_admissible'), ('N19', 'eventless_background_raw'), ('N19', 'eventless_background_admissible'), ('N19', 'event_weight_raw'), ('N19', 'event_weight_admissible')]),
    'bootstrap_10000_seed_18': CONFIG['bootstrap_reps'] == 10000 and CONFIG['bootstrap_seed'] == 18,
    'n21_ordinary_response_contract': bool(n21_compatibility_audit(reference_results['ordinary_response_ratios']).iloc[0]['n21_contract_present']),
    'test_used_for_selection': False, 'full_rediscovery_executed': RUN_FULL_REDISCOVERY, 'execution_errors': 0,
}
ready_for_n21 = all(value is True or value == 0 or value == 'N/A' for key, value in closure_status.items() if key not in {'test_used_for_selection', 'full_rediscovery_executed'}) and not closure_status['test_used_for_selection'] and not closure_status['full_rediscovery_executed']
print('N20 REUSE closure:', 'PASS' if ready_for_n21 else 'FAIL')
if not ready_for_n21:
    raise AssertionError(closure_status)


N20 REUSE closure: PASS


In [17]:
_features = flexible_feature_names(('R', 'I'), 3)
assert _features == ['R_lag3', 'I_lag3', 'R_lag2', 'I_lag2', 'R_lag1', 'I_lag1']
_frozen = frozen_n19_flexible_spec()
assert _frozen['selected'][('R', 'MLP')]['final_epochs'] == 55
assert _frozen['selected'][('I', 'TRANSFORMER')]['final_epochs'] == 61
_specification_smoke = _specification_table(_frozen['selected']); assert set(['target', 'model_family', 'parent_blocks', 'lag_order', 'config_id', 'hyperparameters', 'final_epochs']).issubset(_specification_smoke.columns) and len(_specification_smoke) == 8
_structure_smoke = frozen_n16_structure(); assert set(['selected_orders', 'retained_structure', 'metadata']).issubset(_structure_smoke) and set(_structure_smoke['selected_orders']) == {'R', 'I'}
assert CONFIG['bootstrap_reps'] == 10000 and CONFIG['bootstrap_seed'] == 18
assert _structure_smoke['selected_orders'] == {'R': 1, 'I': 12}
_support_smoke = pd.DataFrame({'R': [1.0], 'I': [1.0], **{f'{state}_lag{lag}': [1.0] for state in CONFIG['states'] for lag in range(1, 13)}}); _support_smoke.loc[0, 'Q_lag1'] = np.nan; assert not _parametric_eligibility(_support_smoke, 'R')[0]; _support_smoke.loc[0, 'Q_lag1'] = 1.0; _support_smoke.loc[0, 'R_lag12'] = np.nan; assert not _parametric_eligibility(_support_smoke, 'I')[0]
assert MLPRegressor(2, [4])(torch.zeros(3, 2)).shape == (3,)
assert TransformerRegressor(2, 1, 16, 2, 32)(torch.zeros(3, 1, 2)).shape == (3,)
assert TransformerRegressor(3, 5, 24, 4, 48)(torch.zeros(3, 5, 3)).shape == (3,)
_tiny_panel = pd.DataFrame({'model_day': pd.to_datetime(['2020-01-02', '2020-01-03']), 'sample_split': ['train', 'test'], 'squared_return': [1.0, 1.0], 'realised_variance_ann_252': [1.0, 1.0], IV_COLUMN: [1.0, 1.0]})
_tiny_events = pd.DataFrame({'event_id': ['overlap_a', 'overlap_b'], 'event_label': ['A', 'B'], 'realised_model_day': [pd.Timestamp('2020-01-02'), pd.Timestamp('2020-01-02')]})
_normalised = _normalise_events(_tiny_events, _tiny_panel, 'overlap smoke')
_normalised_twice = _normalise_events(_normalised, _tiny_panel, 'overlap smoke twice')
assert list(_normalised.filter(regex='^sample_split').columns) == ['sample_split']
assert list(_normalised_twice.filter(regex='^sample_split').columns) == ['sample_split']
assert _normalised[['event_id', 'event_label', 'realised_model_day', 'sample_split']].equals(_normalised_twice[['event_id', 'event_label', 'realised_model_day', 'sample_split']])
_mapped = target_event_mapping(_normalised, _tiny_panel)
assert len(_normalised) == 2 and _mapped['is_overlap'].all() and (_mapped['n_target_occurrences'] == 2).all()
_require_event_subset(_normalised, _normalised.iloc[[0]])
assert len(select_evaluation_events(_normalised, families=['A'])) == 1
_history_mapping_smoke = pd.DataFrame({'target': ['R', 'R'], 'target_model_day': pd.to_datetime(['2020-01-02', '2020-01-03'])})
_evaluation_all_smoke = _history_mapping_smoke.copy()
_evaluation_subset_smoke = _history_mapping_smoke.iloc[[1]].copy()
_masked_history_days = set(_history_mapping_smoke.loc[_history_mapping_smoke['target'].eq('R'), 'target_model_day'])
assert _masked_history_days == set(_history_mapping_smoke.loc[_history_mapping_smoke['target'].eq('R'), 'target_model_day'])
assert set(_evaluation_subset_smoke['target_model_day']).issubset(_masked_history_days)
_mask_panel_smoke = pd.DataFrame({'model_day': pd.date_range('2020-01-01', periods=6, freq='D'), 'R': [1., 2., 3., 5., 8., 13.], 'R_lag1': [0., 1., 2., 3., 5., 8.]})
_mask_spec_smoke = ModelSpec('PARAMETRIC_N17', 'R', tuple(), 1, parametric_features=('R_lag1',))
_mask_calibration = _mask_panel_smoke.loc[~_mask_panel_smoke['model_day'].isin(_masked_history_days)].copy()
_all_forecaster = build_forecaster(_mask_spec_smoke).fit(_mask_calibration)
_subset_forecaster = build_forecaster(_mask_spec_smoke).fit(_mask_calibration)
_common_evaluation_smoke = _mask_panel_smoke.iloc[[4, 5]]
assert np.allclose(_all_forecaster.predict(_common_evaluation_smoke), _subset_forecaster.predict(_common_evaluation_smoke))
assert ALIGNMENT_VERSION == 'MODEL_DAY_1700_NY_SAME_MODEL_DAY' and not bool(spec18['bloomberg_snapshot_time_identified'])
assert reference_mapping['realised_model_day'].eq(reference_mapping['implied_model_day']).all()
assert PUBLIC_API_KEYS <= {'structure', 'parametric_ordinary', 'parametric_eventless', 'flexible_spec', 'flexible_ordinary', 'flexible_eventless', 'event_mapping', 'event_weights', 'pooled_event_weights', 'ordinary_response_ratios', 'model_specification', 'model_specifications', 'ordinary_forecasts', 'eventless_forecasts', 'diagnostics', 'run_metadata'}
_common_forecaster_specs = [
    ModelSpec('PARAMETRIC_N17', 'R', tuple(), 1, parametric_features=('R_lag1',)),
    ModelSpec('RF', 'R', ('R', 'I'), 1, configuration=RF_GRID['RF_01']),
    ModelSpec('GB', 'R', ('R', 'I'), 2, configuration=GB_GRID['GB_01']),
    ModelSpec('MLP', 'R', ('R', 'I'), 1, configuration=MLP_GRID['MLP_01'], final_epochs=1),
    ModelSpec('TRANSFORMER', 'I', ('I', 'Q', 'R'), 5, configuration=TR_GRID['TR_01'], final_epochs=1),
]
for _spec in _common_forecaster_specs:
    _forecaster = build_forecaster(_spec)
    assert callable(_forecaster.fit) and callable(_forecaster.predict)
    assert _forecaster.features == _spec.feature_names
assert n21_compatibility_audit(pd.DataFrame(columns=['model_day', 'sample_split', 'target', 'model', 'actual', 'ordinary_background_forecast', 'ordinary_response_ratio', 'ordinary_background_forecast_raw', 'ordinary_response_ratio_raw', 'admissible_forecast', 'ordinary_response_ratio_adm'])).iloc[0]['n21_contract_present']
print('N20 common-forecaster, dynamic-schema, overlap, and API-contract checks passed.')


N20 common-forecaster, dynamic-schema, overlap, and API-contract checks passed.


In [18]:
# Atomic export cell: production files change only after closure has passed.
if 'ready_for_n21' not in globals():
    raise RuntimeError('Run the closure cell first.')
if ready_for_n21 is not True:
    raise RuntimeError('N20 closure did not pass; production exports were not written.')
export_n20_results(reference_results, equivalence_summary=equivalence_summary)
print('N20 production exports written after successful closure.')


N20 production exports written after successful closure.


# N20 final API summary and use

N20 packages the frozen Steps 1–4 workflow as Exercise 5 software design, rather than adding new mathematics. Parametric OLS, RF, GB, MLP, and Transformer implementations all expose the same fit/predict contract; their family-specific preprocessing and training remain inside the forecaster adapters. Generic background and event-weight functions therefore do not need model-family branches.

REUSE is the normal execution mode. It loads frozen N16/N19 specifications but still refits OLS coefficients, trees, neural weights, and neural scalers for the supplied calibration and event-history sample. FULL can rediscover specifications, but is not run by default. Ordinary/event-unaware and eventless backgrounds are returned separately. Raw positive eventless denominators define diagnostic raw event weights, while canonical production event weights use the frozen target-specific admissible backgrounds. The standardized ordinary-response output remains the N21 hidden-event-discovery input.

A fresh notebook can use the same real API as follows:

```python
config = make_default_config(mode="REUSE")
frozen_structure = frozen_n16_structure()
frozen_models = frozen_n19_flexible_spec()
selected_events = select_evaluation_events(
    event_history,
    families=["BoJ", "FOMC"],
    start_date="2021-11-15",
    end_date="2026-07-02",
)
result = estimate_event_weights(
    model_panel=model_panel,
    event_history=event_history,
    evaluation_events=selected_events,
    structure=frozen_structure,
    flexible_spec=frozen_models,
    config=config,
    mode="REUSE",
)
display(result["pooled_event_weights"])
```

The result contains run_metadata, structure, model_specifications, ordinary_forecasts, eventless_forecasts, event_weights, pooled_event_weights, and diagnostics. N20’s REUSE equivalence audit compares unrounded output with the frozen N17, N18, and N19 exports by stable identifiers before a run can be treated as ready.
